<a href="https://colab.research.google.com/github/tsm-mehmetakiftasoz/tsm_makif/blob/main/%D0%93%D0%BE%D1%82%D0%BE%D0%B2%D0%BE_%D0%92%D1%81%D0%B5%20%D0%B2%20%D0%BE%D0%B4%D0%BD%D0%BE%D0%BC_%20beta_test_ver7-4_ge%C3%A7ici%20haz%C4%B1r.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Предназначено для объединения всех отчетов. Имена должны быть строго идентичными, и при выгрузке из SAP они должны быть в формате MAKIF_REPORT, иначе код не будет работать!!!

Необходимые отчеты:
*   ME5A_2022, ME5A_2023, ME5A_2024, ME5A_2025,ME5A_2026 (могут быть выгружены из SAP в формате 01.01.202X-31.12.202X)
*   ME2N_2022, ME2N_2023, ME2N_2024, ME2N_2025,ME2N_2026 (могут быть выгружены из SAP в формате 01.01.202X-31.12.202X)
*   zmm059_2022, zmm059_2023, zmm059_2025,zmm059_2026, zmm059_2024_1, zmm059_2024_2, zmm059_2024_3, zmm059_2024_4.(Отчеты за 2022, 2023, 2025 годы могут быть выгружены из SAP в формате 01.01.202X-31.12.202X. Для 2024 года граничные даты: 23.04.2024, 15.07.2024, 29.11.2024 и 01.12.2024 по 31.12.2024 (на 30.11 система выдает ошибку)).
*   zmb51_2022, zmb51_2023, zmb51_2025_1 (01.01.25-31.05.25), zmb51_2025_2 (01.06.2025-31.12.2025),zmb51_2026, zmb51_2024_1 (01.01-30.04.24), zmb51_2024_2 (01.05.2024-31.07.2024), zmb51_2024_3 (01.08.2024-30.09.2024), zmb51_2024_4 (01.10.2024-30.11.2024), zmb51_2024_5 (01.12.2024-31.12.2024)
*   ZMM067_2022(01.01.202X-31.12.202X) , ZMM067_2023(01.01.202X-31.12.202X), ZMM067_2024_1(01.01-31.05.24),ZMM067_2024_2(01.06-31.12.24),ZMM067_2025(01.01.202X-31.12.202X),ZMM067_2026

In [1]:
!pip install pandas
!pip install xlsxwriter

In [2]:
import pandas as pd
import numpy as np
from google.colab import files
import os
import zipfile

In [3]:
def duzenle_sayi_sutunu(df, sutun_adi):
    """
    Sayısal değerleri metin olarak tutan ve binlik/ondalık ayracı içeren sütunu dönüştürür.
    - Nokta (.) binlik ayracı olarak kaldırılır
    - Virgül (,) ondalık ayracı olarak noktaya çevrilir
    - Float olarak dönüştürülür
    - Eğer tam sayı ise integer olarak, değilse float olarak kalır
    - Sonuç string olarak döndürülür (görsel temizlik için)

    Parametreler:
    df : pd.DataFrame
    sutun_adi : str

    Dönüş:
    df (işlenmiş hali)
    """
    df[sutun_adi] = (
        df[sutun_adi]
        .astype(str)
        .str.replace(" ", "", regex=False)
        .str.replace(".", "", regex=False)
        .str.replace(",", ".", regex=False)
        .astype(float)
        .apply(lambda x: str(int(x)) if x.is_integer() else str(x))
    )
    return df


In [4]:
def temizle_ve_cevir(x):
    try:
        return str(int(float(str(x).strip())))
    except:
        return str(x).strip()


In [5]:
# Temizlenecek değerler listesi
temizlenecekler = ["nan", "Nan", "NaN", ""," ","0","naN","nAn","NAn","nAN"]

def temizle_nan(value):
    if str(value).strip().lower() in [x.lower() for x in temizlenecekler]:
        return pd.NA
    return value


In [6]:
# Güvenli filtreleme fonksiyonu
def dolu_mi(x):
    return pd.notna(x) and str(x).strip() != "" and str(x).strip().upper() != "<NA>"


In [7]:
#me5a'nın yüklenmesi
df_me5a_2022 = pd.read_csv('/content/ME5A_2022.csv', sep=None, engine='python')
df_me5a_2023 = pd.read_csv('/content/ME5A_2023.csv', sep=None, engine='python')
df_me5a_2024 = pd.read_csv('/content/ME5A_2024.csv', sep=None, engine='python')
df_me5a_2025 = pd.read_csv('/content/ME5A_2025.csv', sep=None, engine='python')
df_me5a_2026 = pd.read_csv('/content/ME5A_2026.csv', sep=None, engine='python')
# sep=None will automatically detect the delimiter
# engine='python' is slower but more robust for complex cases
#zmm059un yüklenmesi
df_zmm059_2022 = pd.read_csv('/content/zmm059_2022.csv', sep=None, engine='python')
df_zmm059_2023 = pd.read_csv('/content/zmm059_2023.csv', sep=None, engine='python')
df_zmm059_2024_1 = pd.read_csv('/content/zmm059_2024_1.csv', sep=None, engine='python')
df_zmm059_2024_2 = pd.read_csv('/content/zmm059_2024_2.csv', sep=None, engine='python')
df_zmm059_2024_3 = pd.read_csv('/content/zmm059_2024_3.csv', sep=None, engine='python')
df_zmm059_2024_4 = pd.read_csv('/content/zmm059_2024_4.csv', sep=None, engine='python')
df_zmm059_2025 = pd.read_csv('/content/zmm059_2025.csv', sep=None, engine='python')
df_zmm059_2026 = pd.read_csv('/content/zmm059_2026.csv', sep=None, engine='python')
df_zmm059_2026_1 = pd.read_csv('/content/zmm059_2026_1.csv', sep=None, engine='python')
#yardımcı tabloların yüklenmesi
df_tedarikci_listesi = pd.read_csv('/content/tedarikci_isim.csv', sep=None, engine='python')
df_material_list = pd.read_csv('/content/material_list.csv', sep=None, engine='python')
# sep=None will automatically detect the delimiter
# engine='python' is slower but more robust for complex cases
#me2n'in yüklenmesi
df_me2n_2022 = pd.read_csv('/content/ME2N_2022.csv', sep=None, engine='python', dtype={"Единица цены":object})
df_me2n_2023 = pd.read_csv('/content/ME2N_2023.csv', sep=None, engine='python', dtype={"Единица цены":object})
df_me2n_2024 = pd.read_csv('/content/ME2N_2024.csv', sep=None, engine='python', dtype={"Единица цены":object})
df_me2n_2025 = pd.read_csv('/content/ME2N_2025.csv', sep=None, engine='python', dtype={"Единица цены":object})
df_me2n_2026 = pd.read_csv('/content/ME2N_2026.csv', sep=None, engine='python', dtype={"Единица цены":object})
# sep=None will automatically detect the delimiter
# engine='python' is slower but more robust for complex cases
#zmb51'in yüklenmesi
df_zmb51_2022 = pd.read_csv('/content/zmb51_2022.csv', sep=None, engine='python')
df_zmb51_2023 = pd.read_csv('/content/zmb51_2023.csv', sep=None, engine='python')
df_zmb51_2024_1 = pd.read_csv('/content/zmb51_2024_1.csv', sep=None, engine='python')
df_zmb51_2024_2 = pd.read_csv('/content/zmb51_2024_2.csv', sep=None, engine='python')
df_zmb51_2024_3 = pd.read_csv('/content/zmb51_2024_3.csv', sep=None, engine='python')
df_zmb51_2024_4 = pd.read_csv('/content/zmb51_2024_4.csv', sep=None, engine='python')
df_zmb51_2024_5 = pd.read_csv('/content/zmb51_2024_5.csv', sep=None, engine='python')
#df_zmb51_2024_6 = pd.read_csv('/content/zmb51_2024_6.csv', sep=None, engine='python')
#df_zmb51_2024_7 = pd.read_csv('/content/zmb51_2024_7.csv', sep=None, engine='python')
df_zmb51_2025 = pd.read_csv('/content/zmb51_2025.csv', sep=None, engine='python')
df_zmb51_2025_2 = pd.read_csv('/content/zmb51_2025-2.csv', sep=None, engine='python')
df_zmb51_2026 = pd.read_csv('/content/zmb51_2026.csv', sep=None, engine='python')
# sep=None will automatically detect the delimiter
# engine='python' is slower but more robust for complex cases
#zmm067'in yüklenmesi
df_zmm067_2022 = pd.read_csv('/content/ZMM067_2022.csv', sep=None, engine='python')
df_zmm067_2023 = pd.read_csv('/content/ZMM067_2023.csv', sep=None, engine='python')
df_zmm067_2024_1 = pd.read_csv('/content/ZMM067_2024_1.csv', sep=None, engine='python')
df_zmm067_2024_2 = pd.read_csv('/content/ZMM067_2024_2.csv', sep=None, engine='python')
df_zmm067_2025 = pd.read_csv('/content/ZMM067_2025.csv', sep=None, engine='python')
df_zmm067_2026 = pd.read_csv('/content/ZMM067_2026.csv', sep=None, engine='python')
#
#zpp001 yüklenmesi
zpp001 = pd.read_csv('/content/ZPP001.csv', sep=None, engine='python')



In [8]:
#ilk sutuna yıl yazdırma me5a
df_me5a_2022["ГОД"] = "2022"
cols = ["ГОД"] + [col for col in df_me5a_2022.columns if col != "ГОД"]
df_me5a_2022 = df_me5a_2022[cols]
#
df_me5a_2023["ГОД"] = "2023"
cols = ["ГОД"] + [col for col in df_me5a_2023.columns if col != "ГОД"]
df_me5a_2023 = df_me5a_2023[cols]
#
df_me5a_2024["ГОД"] = "2024"
cols = ["ГОД"] + [col for col in df_me5a_2024.columns if col != "ГОД"]
df_me5a_2024 = df_me5a_2024[cols]
#
df_me5a_2025["ГОД"] = "2025"
cols = ["ГОД"] + [col for col in df_me5a_2025.columns if col != "ГОД"]
df_me5a_2025 = df_me5a_2025[cols]
#
df_me5a_2026["ГОД"] = "2026"
cols = ["ГОД"] + [col for col in df_me5a_2026.columns if col != "ГОД"]
df_me5a_2026 = df_me5a_2026[cols]

#ilk sutuna yıl yazdırma zmm059
df_zmm059_2022["ГОД"] = "2022"
cols = ["ГОД"] + [col for col in df_zmm059_2022.columns if col != "ГОД"]
df_zmm059_2022 = df_zmm059_2022[cols]
#
df_zmm059_2023["ГОД"] = "2023"
cols = ["ГОД"] + [col for col in df_zmm059_2023.columns if col != "ГОД"]
df_zmm059_2023 = df_zmm059_2023[cols]
#
df_zmm059_2025["ГОД"] = "2025"
cols = ["ГОД"] + [col for col in df_zmm059_2025.columns if col != "ГОД"]
df_zmm059_2025 = df_zmm059_2025[cols]
#
df_zmm059_2024_1["ГОД"] = "2024"
cols = ["ГОД"] + [col for col in df_zmm059_2024_1.columns if col != "ГОД"]
df_zmm059_2024_1 = df_zmm059_2024_1[cols]
#
df_zmm059_2024_2["ГОД"] = "2024"
cols = ["ГОД"] + [col for col in df_zmm059_2024_2.columns if col != "ГОД"]
df_zmm059_2024_2 = df_zmm059_2024_2[cols]
#
df_zmm059_2024_3["ГОД"] = "2024"
cols = ["ГОД"] + [col for col in df_zmm059_2024_3.columns if col != "ГОД"]
df_zmm059_2024_3 = df_zmm059_2024_3[cols]
#
df_zmm059_2024_4["ГОД"] = "2024"
cols = ["ГОД"] + [col for col in df_zmm059_2024_4.columns if col != "ГОД"]
df_zmm059_2024_4 = df_zmm059_2024_4[cols]
#
df_zmm059_2026["ГОД"] = "2026"
cols = ["ГОД"] + [col for col in df_zmm059_2026.columns if col != "ГОД"]
df_zmm059_2026 = df_zmm059_2026[cols]
#
df_zmm059_2026_1["ГОД"] = "2026"
cols = ["ГОД"] + [col for col in df_zmm059_2026_1.columns if col != "ГОД"]
df_zmm059_2026_1 = df_zmm059_2026_1[cols]

#ilk sutuna yıl yazdırma me2n
df_me2n_2022["ГОД"] = "2022"
cols = ["ГОД"] + [col for col in df_me2n_2022.columns if col != "ГОД"]
df_me2n_2022 = df_me2n_2022[cols]
#
df_me2n_2023["ГОД"] = "2023"
cols = ["ГОД"] + [col for col in df_me2n_2023.columns if col != "ГОД"]
df_me2n_2023 = df_me2n_2023[cols]
#
df_me2n_2024["ГОД"] = "2024"
cols = ["ГОД"] + [col for col in df_me2n_2024.columns if col != "ГОД"]
df_me2n_2024 = df_me2n_2024[cols]
#
df_me2n_2025["ГОД"] = "2025"
cols = ["ГОД"] + [col for col in df_me2n_2025.columns if col != "ГОД"]
df_me2n_2025 = df_me2n_2025[cols]
#
df_me2n_2026["ГОД"] = "2026"
cols = ["ГОД"] + [col for col in df_me2n_2026.columns if col != "ГОД"]
df_me2n_2026 = df_me2n_2026[cols]
#ilk sutuna yıl yazdırma zmb51
df_zmb51_2022["ГОД"] = "2022"
cols = ["ГОД"] + [col for col in df_zmb51_2022.columns if col != "ГОД"]
df_zmb51_2022 = df_zmb51_2022[cols]
#
df_zmb51_2023["ГОД"] = "2023"
cols = ["ГОД"] + [col for col in df_zmb51_2023.columns if col != "ГОД"]
df_zmb51_2023 = df_zmb51_2023[cols]
#
df_zmb51_2024_1["ГОД"] = "2024"
cols = ["ГОД"] + [col for col in df_zmb51_2024_1.columns if col != "ГОД"]
df_zmb51_2024_1 = df_zmb51_2024_1[cols]
#
df_zmb51_2024_2["ГОД"] = "2024"
cols = ["ГОД"] + [col for col in df_zmb51_2024_2.columns if col != "ГОД"]
df_zmb51_2024_2 = df_zmb51_2024_2[cols]
#
df_zmb51_2024_3["ГОД"] = "2024"
cols = ["ГОД"] + [col for col in df_zmb51_2024_3.columns if col != "ГОД"]
df_zmb51_2024_3 = df_zmb51_2024_3[cols]
#
df_zmb51_2024_4["ГОД"] = "2024"
cols = ["ГОД"] + [col for col in df_zmb51_2024_4.columns if col != "ГОД"]
df_zmb51_2024_4 = df_zmb51_2024_4[cols]
#
df_zmb51_2024_5["ГОД"] = "2024"
cols = ["ГОД"] + [col for col in df_zmb51_2024_5.columns if col != "ГОД"]
df_zmb51_2024_5 = df_zmb51_2024_5[cols]
#
df_zmb51_2025["ГОД"] = "2025"
cols = ["ГОД"] + [col for col in df_zmb51_2025.columns if col != "ГОД"]
df_zmb51_2025 = df_zmb51_2025[cols]
#
df_zmb51_2025_2["ГОД"] = "2025"
cols = ["ГОД"] + [col for col in df_zmb51_2025_2.columns if col != "ГОД"]
df_zmb51_2025_2 = df_zmb51_2025_2[cols]
#
df_zmb51_2026["ГОД"] = "2026"
cols = ["ГОД"] + [col for col in df_zmb51_2026.columns if col != "ГОД"]
df_zmb51_2026 = df_zmb51_2026[cols]
#ilk sutuna yıl yazdırma zmm067
# 2022
df_zmm067_2022["ГОД"] = "2022"
cols = ["ГОД"] + [col for col in df_zmm067_2022.columns if col != "ГОД"]
df_zmm067_2022 = df_zmm067_2022[cols]

# 2023
df_zmm067_2023["ГОД"] = "2023"
cols = ["ГОД"] + [col for col in df_zmm067_2023.columns if col != "ГОД"]
df_zmm067_2023 = df_zmm067_2023[cols]

# 2024_1
df_zmm067_2024_1["ГОД"] = "2024"
cols = ["ГОД"] + [col for col in df_zmm067_2024_1.columns if col != "ГОД"]
df_zmm067_2024_1 = df_zmm067_2024_1[cols]

# 2024_2
df_zmm067_2024_2["ГОД"] = "2024"
cols = ["ГОД"] + [col for col in df_zmm067_2024_2.columns if col != "ГОД"]
df_zmm067_2024_2 = df_zmm067_2024_2[cols]

# 2025
df_zmm067_2025["ГОД"] = "2025"
cols = ["ГОД"] + [col for col in df_zmm067_2025.columns if col != "ГОД"]
df_zmm067_2025 = df_zmm067_2025[cols]
# 2026
df_zmm067_2026["ГОД"] = "2026"
cols = ["ГОД"] + [col for col in df_zmm067_2026.columns if col != "ГОД"]
df_zmm067_2026 = df_zmm067_2026[cols]

In [9]:
#me5a birleştirme
df_listesi_me5a = [df_me5a_2026,df_me5a_2025,df_me5a_2024, df_me5a_2023,df_me5a_2022 ]
df_me5a_2022_2025 = pd.concat(df_listesi_me5a, ignore_index=True)
# zmm059 birleştirme
df_listesi_zmm059 = [df_zmm059_2026_1,df_zmm059_2026,df_zmm059_2025, df_zmm059_2024_4, df_zmm059_2024_3, df_zmm059_2024_2, df_zmm059_2024_1, df_zmm059_2023,df_zmm059_2022 ]
df_zmm059_2022_2025 = pd.concat(df_listesi_zmm059, ignore_index=True)
#me2n birleştirme
df_listesi_me2n = [df_me2n_2026,df_me2n_2025, df_me2n_2024, df_me2n_2023,df_me2n_2022]
df_me2n_2022_2025 = pd.concat(df_listesi_me2n, ignore_index=True)
#zmb51 birleştirme
df_listesi_zmb51 = [df_zmb51_2026,df_zmb51_2025_2,df_zmb51_2025,df_zmb51_2024_5,df_zmb51_2024_4,df_zmb51_2024_3,
                    df_zmb51_2024_2,df_zmb51_2024_1,df_zmb51_2023,df_zmb51_2022
    ]
birlesik_df_zmb51 = pd.concat(df_listesi_zmb51, ignore_index=True)
#zmm067 birleştirme
df_listesi_zmm067 = [df_zmm067_2026,df_zmm067_2025,df_zmm067_2024_2,df_zmm067_2024_1,df_zmm067_2023,df_zmm067_2022]
df_zmm067_2022_2025 = pd.concat(df_listesi_zmm067, ignore_index=True)

In [10]:
#FBL1N YÜKLEMEK
df_fbl1n_2022 = pd.read_csv('/content/FBL1N 2022.csv', sep=None, engine='python')
df_fbl1n_2023_1 = pd.read_csv('/content/FBL1N 2023_1.csv', sep=None, engine='python')
df_fbl1n_2023_2 = pd.read_csv('/content/FBL1N 2023_2.csv', sep=None, engine='python')
df_fbl1n_2024 = pd.read_csv('/content/FBL1N 2024.csv', sep=None, engine='python')
df_fbl1n_2025 = pd.read_csv('/content/FBL1N 2025.csv', sep=None, engine='python')
df_fbl1n_2026 = pd.read_csv('/content/FBL1N 2026.csv', sep=None, engine='python')
#MIR5 Yüklemek
df_mir5_2022 = pd.read_csv('/content/MIR5 2022.csv', sep=None, engine='python')
df_mir5_2023 = pd.read_csv('/content/MIR5 2023.csv', sep=None, engine='python')
df_mir5_2024 = pd.read_csv('/content/MIR5 2024.csv', sep=None, engine='python')
df_mir5_2025 = pd.read_csv('/content/MIR5 2025.csv', sep=None, engine='python')
df_mir5_2026 = pd.read_csv('/content/MIR5 2026.csv', sep=None, engine='python')
#ZMM070 Yüklemek
df_zmm070_2024_02 = pd.read_csv('/content/ZMM070 2024_02.csv', sep=None, engine='python')
df_zmm070_2024_03 = pd.read_csv('/content/ZMM070 2024_03.csv', sep=None, engine='python')
df_zmm070_2024_04 = pd.read_csv('/content/ZMM070 2024_04.csv', sep=None, engine='python')
df_zmm070_2024_05 = pd.read_csv('/content/ZMM070 2024_05.csv', sep=None, engine='python')
df_zmm070_2024_06 = pd.read_csv('/content/ZMM070 2024_06.csv', sep=None, engine='python')
df_zmm070_2024_07 = pd.read_csv('/content/ZMM070 2024_07.csv', sep=None, engine='python')
df_zmm070_2024_10 = pd.read_csv('/content/ZMM070 2024_10.csv', sep=None, engine='python')
df_zmm070_2024_11 = pd.read_csv('/content/ZMM070 2024_11.csv', sep=None, engine='python')
df_zmm070_2025_1 = pd.read_csv('/content/ZMM070 2025_01.csv', sep=None, engine='python')
df_zmm070_2025_2 = pd.read_csv('/content/ZMM070 2025_02.csv', sep=None, engine='python')
#df_zmm070_2025_3_1 = pd.read_csv('/content/ZMM070 2025_03-1.csv', sep=None, engine='python')
df_zmm070_2025_3_2 = pd.read_csv('/content/ZMM070 2025_03-2.csv', sep=None, engine='python')
df_zmm070_2025_3_3 = pd.read_csv('/content/ZMM070 2025_03-3.csv', sep=None, engine='python')
df_zmm070_2025_4 = pd.read_csv('/content/ZMM070 2025_04.csv', sep=None, engine='python')
df_zmm070_2025_5 = pd.read_csv('/content/ZMM070 2025_05.csv', sep=None, engine='python')
df_zmm070_2025_6 = pd.read_csv('/content/ZMM070 2025_06.csv', sep=None, engine='python')
df_zmm070_2025_7 = pd.read_csv('/content/ZMM070 2025_07.csv', sep=None, engine='python')
#df_zmm070_2025_8_1 = pd.read_csv('/content/ZMM070 2025_03-2.csv', sep=None, engine='python')
df_zmm070_2025_8_2 = pd.read_csv('/content/ZMM070 2025_08-2.csv', sep=None, engine='python')
df_zmm070_2025_8_3 = pd.read_csv('/content/ZMM070 2025_08-3.csv', sep=None, engine='python')
#df_zmm070_2025_9_1 = pd.read_csv('/content/ZMM070 2025_03-2.csv', sep=None, engine='python')
df_zmm070_2025_9_2 = pd.read_csv('/content/ZMM070 2025_09-2.csv', sep=None, engine='python')
df_zmm070_2025_9_3 = pd.read_csv('/content/ZMM070 2025_09-3.csv', sep=None, engine='python')
df_zmm070_2025_10_1 = pd.read_csv('/content/ZMM070 2025_10-1.csv', sep=None, engine='python')
df_zmm070_2025_10_2 = pd.read_csv('/content/ZMM070 2025_10-2.csv', sep=None, engine='python')
df_zmm070_2025_11 = pd.read_csv('/content/ZMM070 2025_11.csv', sep=None, engine='python')
#df_zmm070_2025_12_1 = pd.read_csv('/content/ZMM070 2025_03-2.csv', sep=None, engine='python')
df_zmm070_2025_12_2 = pd.read_csv('/content/ZMM070 2025_12-2.csv', sep=None, engine='python')
df_zmm070_2025_12_3 = pd.read_csv('/content/ZMM070 2025_12-3.csv', sep=None, engine='python')
df_zmm070_2026_01 = pd.read_csv('/content/ZMM070 2026_01.csv', sep=None, engine='python')
df_zmm070_2026_02 = pd.read_csv('/content/ZMM070 2026_02.csv', sep=None, engine='python')
df_zmm070_2026_03 = pd.read_csv('/content/ZMM070 2026_03.csv', sep=None, engine='python')

In [11]:
#fbl1n birleştirme
df_listesi_fbl1n = [df_fbl1n_2026,df_fbl1n_2025,df_fbl1n_2024,df_fbl1n_2023_2,df_fbl1n_2023_1,df_fbl1n_2022 ]
df_fbl1n_2022_2025 = pd.concat(df_listesi_fbl1n, ignore_index=True)
#MIR5 birleştirme
df_listesi_mir5 = [df_mir5_2026,df_mir5_2025,df_mir5_2024,df_mir5_2023,df_mir5_2022 ]
df_mir5_2022_2025 = pd.concat(df_listesi_mir5, ignore_index=True)
#ZMM070 birleştirme
df_listesi_zmm070 = [df_zmm070_2026_03,df_zmm070_2026_02,df_zmm070_2026_01,df_zmm070_2025_12_3,df_zmm070_2025_12_2,df_zmm070_2025_11,df_zmm070_2025_10_2,df_zmm070_2025_10_1,
     df_zmm070_2025_9_3, df_zmm070_2025_9_2,df_zmm070_2025_8_3,df_zmm070_2025_8_2,df_zmm070_2025_7,df_zmm070_2025_6,df_zmm070_2025_5,df_zmm070_2025_4,
    df_zmm070_2025_3_3,df_zmm070_2025_3_2,df_zmm070_2025_2,df_zmm070_2025_1,df_zmm070_2024_06,df_zmm070_2024_11,df_zmm070_2024_03,df_zmm070_2024_02,df_zmm070_2024_07,df_zmm070_2024_10,df_zmm070_2024_04]
df_zmm070_2022_2025 = pd.concat(df_listesi_zmm070, ignore_index=True)
df_zmm070_2022_2025 = df_zmm070_2022_2025.drop_duplicates()

In [12]:
#yedek tablolar
df_me5a_2022_2025_yedek=df_me5a_2022_2025.copy()
df_zmm059_2022_2025_yedek=df_zmm059_2022_2025.copy()
df_me2n_2022_2025_yedek=df_me2n_2022_2025.copy()
birlesik_df_zmb51_yedek=birlesik_df_zmb51.copy()
df_zmm067_2022_2025_yedek=df_zmm067_2022_2025.copy()

In [13]:
zpp001["fat_ted"]=zpp001["Инвойс"]+"-"+zpp001["Кредитор"]
df_fbl1n_2022_2025["﻿Ссылка на счет"] = df_fbl1n_2022_2025["﻿Ссылка на счет"].apply(temizle_ve_cevir)
df_fbl1n_2022_2025["Кредитор"] = df_fbl1n_2022_2025["Кредитор"].apply(temizle_ve_cevir)
df_fbl1n_2022_2025["fat_ted"]=df_fbl1n_2022_2025["﻿Ссылка на счет"]+"-"+df_fbl1n_2022_2025["Кредитор"]

In [14]:
# 1. Anahtar sütunları tuple haline getirelim
df_fbl1n_2022_2025["key"] = df_fbl1n_2022_2025[["fat_ted"]].apply(tuple, axis=1)
zpp001["key"] = zpp001[["fat_ted"]].apply(tuple, axis=1)

# 2. Aktarılacak sütunlar ve hedef adları
map_dict_B3_status = {
    "Ссылка":"Номер_счета",
    "Дата документа":"Дата_документа_счета",

}

# 3. Her sütun için dict map uygulama
for source_col, target_col in map_dict_B3_status.items():
    temp_dict = dict(zip(df_fbl1n_2022_2025["key"], df_fbl1n_2022_2025[source_col]))
    zpp001[target_col] = zpp001["key"].map(temp_dict)

# 4. Geçici key sütunlarını kaldır
zpp001.drop(columns=["key"], inplace=True)
df_fbl1n_2022_2025.drop(columns=["key"], inplace=True)

In [15]:
df_zmm070_2022_2025["Документ закупки"] = df_zmm070_2022_2025["Документ закупки"].apply(temizle_ve_cevir)
df_zmm070_2022_2025["Позиция"] = df_zmm070_2022_2025["Позиция"].apply(temizle_ve_cevir)

In [16]:
# 1. Anahtar sütunları tuple haline getirelim
df_zmm070_2022_2025["key"] = df_zmm070_2022_2025[["Ссылка", "Выставитель счета"]].apply(tuple, axis=1)
zpp001["key"] = zpp001[["Номер_счета", "Кредитор"]].apply(tuple, axis=1)

# 2. Aktarılacak sütunlar ve hedef adları
map_dict = {
    "Документ закупки":"Документ закупки",
    "Позиция":"зп_поз"


}

# 3. Her sütun için dict map uygulama
for source_col, target_col in map_dict.items():
    temp_dict = dict(zip(df_zmm070_2022_2025["key"], df_zmm070_2022_2025[source_col]))
    zpp001[target_col] = zpp001["key"].map(temp_dict)

# 4. Geçici key sütunlarını kaldır
zpp001.drop(columns=["key"], inplace=True)
df_zmm070_2022_2025.drop(columns=["key"], inplace=True)


In [17]:
#zmm059
df_zmm059_2022_2025["﻿Документ закупки"] = df_zmm059_2022_2025["﻿Документ закупки"].apply(temizle_ve_cevir)
df_zmm059_2022_2025.rename(columns={'﻿Документ закупки': 'Документ_закупки'}, inplace=True)
df_zmm059_2022_2025["Материал"] = df_zmm059_2022_2025["Материал"].apply(temizle_ve_cevir)
df_zmm059_2022_2025["Заявка"] = df_zmm059_2022_2025["Заявка"].apply(temizle_ve_cevir)
df_zmm059_2022_2025 = duzenle_sayi_sutunu(df_zmm059_2022_2025, "Количество ЗП")
df_zmm059_2022_2025 = duzenle_sayi_sutunu(df_zmm059_2022_2025, "Минимальная цена ЗП")
df_zmm059_2022_2025 = duzenle_sayi_sutunu(df_zmm059_2022_2025, "Единица ЗП")
df_zmm059_2022_2025 = duzenle_sayi_sutunu(df_zmm059_2022_2025, "Цена нетто ЗП")
df_zmm059_2022_2025 = duzenle_sayi_sutunu(df_zmm059_2022_2025, "Итого цена нетто")
df_zmm059_2022_2025["Дата создания"] = pd.to_datetime(df_zmm059_2022_2025["Дата создания"], format="%d.%m.%Y", errors="coerce").dt.date
df_zmm059_2022_2025["Дата поставки"] = pd.to_datetime(df_zmm059_2022_2025["Дата поставки"], format="%d.%m.%Y", errors="coerce").dt.date
df_zmm059_2022_2025["Дата доставки, одобренная Поставщиком"] = pd.to_datetime(df_zmm059_2022_2025["Дата доставки, одобренная Поставщиком"], format="%d.%m.%Y", errors="coerce").dt.date
df_zmm059_2022_2025["Группа закупок"] = df_zmm059_2022_2025["Группа закупок"].apply(temizle_ve_cevir)
df_zmm059_2022_2025["БЕ"] = df_zmm059_2022_2025["БЕ"].apply(temizle_ve_cevir)
df_zmm059_2022_2025["Закуп. организация"] = df_zmm059_2022_2025["Закуп. организация"].apply(temizle_ve_cevir)
df_zmm059_2022_2025["Поставщик"] = df_zmm059_2022_2025["Поставщик"].apply(temizle_ve_cevir)
df_zmm059_2022_2025["Дата создания.1"] = pd.to_datetime(df_zmm059_2022_2025["Дата создания.1"], format="%d.%m.%Y", errors="coerce").dt.date
df_zmm059_2022_2025 = duzenle_sayi_sutunu(df_zmm059_2022_2025, "Цена заказа нетто")
df_zmm059_2022_2025 = duzenle_sayi_sutunu(df_zmm059_2022_2025, "Стоимость брутто")
df_zmm059_2022_2025 = duzenle_sayi_sutunu(df_zmm059_2022_2025, "Недопоставленное количество")
df_zmm059_2022_2025 = duzenle_sayi_sutunu(df_zmm059_2022_2025, "Количество поставки")
df_zmm059_2022_2025 = duzenle_sayi_sutunu(df_zmm059_2022_2025, "Сумма брутто")
df_zmm059_2022_2025 = duzenle_sayi_sutunu(df_zmm059_2022_2025, "Количество по счету")
df_zmm059_2022_2025["Дата проводки"] = pd.to_datetime(df_zmm059_2022_2025["Дата проводки"], format="%d.%m.%Y", errors="coerce").dt.date
df_zmm059_2022_2025 = duzenle_sayi_sutunu(df_zmm059_2022_2025, "Количество по счету.1")
df_zmm059_2022_2025 = duzenle_sayi_sutunu(df_zmm059_2022_2025, "Остаток количества по счету")
df_zmm059_2022_2025["Дата контракта"] = pd.to_datetime(df_zmm059_2022_2025["Дата контракта"], format="%d.%m.%Y", errors="coerce").dt.date
df_zmm059_2022_2025["Дата вступления в силу контракта"] = pd.to_datetime(df_zmm059_2022_2025["Дата вступления в силу контракта"], format="%d.%m.%Y", errors="coerce").dt.date
df_zmm059_2022_2025["Дата окончания действия договора"] = pd.to_datetime(df_zmm059_2022_2025["Дата окончания действия договора"], format="%d.%m.%Y", errors="coerce").dt.date


In [18]:
df_me5a_2022_2025.columns

Index(['ГОД', '﻿Заявка', 'Позиция заявки', 'Материал', 'Индикатор удаления',
       'Затребовал', 'Индик. выдачи', 'Группа деблокир.',
       'Статус обработки заявки', 'Дата заявки', 'Группа закупок',
       'Дата изменения', 'Заказ на поставку', 'Позиция ЗкзНаПостав',
       'Дата заказа', 'Долгосрочный договор', 'ЗатребованКолич',
       'Заказанное к-во', 'Единица измерения', 'RU Наименование',
       'TR Наименование', 'Инвентарный номер', 'Краткий текст',
       'Закуп. организация', 'Создал', 'ПоступлМатериала', 'Поступление счета',
       'Номер проекта', 'Группа материалов', 'Дата поставки',
       'Завод-поставщик', 'Вид документа', 'Статус обработки',
       'Индикатор создания', 'Стратегия деблокир.', 'НДП-материал',
       'Номер потребности', 'Объем дефицита', 'Тип даты поставки',
       'Дата деблокирования'],
      dtype='object')

In [19]:
#ME5A
df_me5a_2022_2025["﻿Заявка"] = df_me5a_2022_2025["﻿Заявка"].apply(temizle_ve_cevir)
df_me5a_2022_2025.rename(columns={'﻿Заявка': 'Заявка'}, inplace=True)
df_me5a_2022_2025.rename(columns={'Индикатор удаления': 'Индикатор удаления_B3'}, inplace=True)
df_me5a_2022_2025["Материал"] = df_me5a_2022_2025["Материал"].apply(temizle_ve_cevir)
df_me5a_2022_2025["'Дата заявки"] = pd.to_datetime(df_me5a_2022_2025["Дата заявки"], format="%d.%m.%Y", errors="coerce").dt.date
df_me5a_2022_2025["Группа закупок"] = df_me5a_2022_2025["Группа закупок"].apply(temizle_ve_cevir)
df_me5a_2022_2025["'Дата заявки"] = pd.to_datetime(df_me5a_2022_2025["Дата заявки"], format="%d.%m.%Y", errors="coerce").dt.date
df_me5a_2022_2025["Дата изменения"] = df_me5a_2022_2025["Дата изменения"].apply(temizle_ve_cevir)
df_me5a_2022_2025["Заказ на поставку"] = df_me5a_2022_2025["Заказ на поставку"].apply(temizle_ve_cevir)
df_me5a_2022_2025["Позиция ЗкзНаПостав"] = df_me5a_2022_2025["Позиция ЗкзНаПостав"].apply(temizle_ve_cevir)
df_me5a_2022_2025 = duzenle_sayi_sutunu(df_me5a_2022_2025, "ЗатребованКолич")
df_me5a_2022_2025 = duzenle_sayi_sutunu(df_me5a_2022_2025, "Заказанное к-во")
df_me5a_2022_2025["Закуп. организация"] = df_me5a_2022_2025["Закуп. организация"].apply(temizle_ve_cevir)
df_me5a_2022_2025["Номер проекта"] = df_me5a_2022_2025["Номер проекта"].apply(temizle_ve_cevir)
df_me5a_2022_2025["Дата деблокирования"] = pd.to_datetime(df_me5a_2022_2025["Дата деблокирования"], format="%d.%m.%Y", errors="coerce").dt.date


In [20]:
df_me5a_cleaned =df_me5a_2022_2025.copy()

In [21]:
#me2n
df_me2n_2022_2025["﻿Документ закупки"] = df_me2n_2022_2025["﻿Документ закупки"].apply(temizle_ve_cevir)
df_me2n_2022_2025.rename(columns={'﻿Документ закупки': 'Документ закупки'}, inplace=True)
df_me2n_2022_2025["Дата документа"] = pd.to_datetime(df_me2n_2022_2025["Дата документа"], format="%d.%m.%Y", errors="coerce").dt.date
df_me2n_2022_2025["Заявка"] = df_me2n_2022_2025["Заявка"].apply(temizle_ve_cevir)
df_me2n_2022_2025["Материал"] = df_me2n_2022_2025["Материал"].apply(temizle_ve_cevir)
df_me2n_2022_2025 = duzenle_sayi_sutunu(df_me2n_2022_2025, "Объем заказа")
df_me2n_2022_2025 = duzenle_sayi_sutunu(df_me2n_2022_2025, "еще поставить (количество)")
df_me2n_2022_2025 = duzenle_sayi_sutunu(df_me2n_2022_2025, "Количество в СЕИ")
df_me2n_2022_2025 = duzenle_sayi_sutunu(df_me2n_2022_2025, "Цена нетто")
df_me2n_2022_2025 = duzenle_sayi_sutunu(df_me2n_2022_2025, "Единица цены")
df_me2n_2022_2025 = duzenle_sayi_sutunu(df_me2n_2022_2025, "СтоимЗаказа нетто")
df_me2n_2022_2025 = duzenle_sayi_sutunu(df_me2n_2022_2025, "Еще для поставки (стоимость)")
df_me2n_2022_2025["Дата поставки"] = pd.to_datetime(df_me2n_2022_2025["Дата поставки"], format="%d.%m.%Y", errors="coerce").dt.date


In [22]:
#zmb51
birlesik_df_zmb51 = duzenle_sayi_sutunu(birlesik_df_zmb51, "Сумма во ВВ")
birlesik_df_zmb51["Партия"] = birlesik_df_zmb51["Партия"].apply(temizle_ve_cevir)
birlesik_df_zmb51["Материал"] = birlesik_df_zmb51["Материал"].apply(temizle_ve_cevir)
birlesik_df_zmb51["Дата ввода"] = pd.to_datetime(birlesik_df_zmb51["Дата ввода"], format="%d.%m.%Y", errors="coerce").dt.date
birlesik_df_zmb51["Дата документа"] = pd.to_datetime(birlesik_df_zmb51["Дата документа"], format="%d.%m.%Y", errors="coerce").dt.date
birlesik_df_zmb51["Дата проводки"] = pd.to_datetime(birlesik_df_zmb51["Дата проводки"], format="%d.%m.%Y", errors="coerce").dt.date
birlesik_df_zmb51["Внутренний заказ"] = birlesik_df_zmb51["Внутренний заказ"].apply(temizle_ve_cevir)
birlesik_df_zmb51["Заказ на поставку"] = birlesik_df_zmb51["Заказ на поставку"].apply(temizle_ve_cevir)
birlesik_df_zmb51["Позиция"] = birlesik_df_zmb51["Позиция"].apply(temizle_ve_cevir)
birlesik_df_zmb51["Документ материала"] = birlesik_df_zmb51["Документ материала"].apply(temizle_ve_cevir)
birlesik_df_zmb51["Поз. док. материала"] = birlesik_df_zmb51["Поз. док. материала"].apply(temizle_ve_cevir)
birlesik_df_zmb51 = duzenle_sayi_sutunu(birlesik_df_zmb51, "Кол-во в ЕИ ввода")
birlesik_df_zmb51 = duzenle_sayi_sutunu(birlesik_df_zmb51, "Количество")
birlesik_df_zmb51['Поставщик'] = birlesik_df_zmb51['Поставщик'].apply(temizle_ve_cevir)
birlesik_df_zmb51.rename(columns={'﻿Вид движения': 'Вид движения'}, inplace=True)


In [23]:
#zmm067
df_zmm067_2022_2025["﻿Заявка"] = df_zmm067_2022_2025["﻿Заявка"].apply(temizle_ve_cevir)
df_zmm067_2022_2025.rename(columns={'﻿Заявка': 'Заявка'}, inplace=True)
df_zmm067_2022_2025["Позиция заявки"] = df_zmm067_2022_2025["Позиция заявки"].apply(temizle_ve_cevir)
df_zmm067_2022_2025["Материал"] = df_zmm067_2022_2025["Материал"].apply(temizle_ve_cevir)
df_zmm067_2022_2025["Заказ на поставку"] = df_zmm067_2022_2025["Заказ на поставку"].apply(temizle_ve_cevir)
df_zmm067_2022_2025["Позиция ЗкзНаПостав"] = df_zmm067_2022_2025["Позиция ЗкзНаПостав"].apply(temizle_ve_cevir)



In [24]:
#df_zmm067_2022_2025["Позиция ЗкзНаПостав"] = df_zmm067_2022_2025["Позиция ЗкзНаПостав"].apply(temizle_ve_cevir)


In [25]:
#df_zmm067_2022_2025["Дополнительное"].sample(n=5)

In [26]:
#df_zmm067_2022_2025.columns

In [27]:
'''
df_me5a_2022_2025 = df_me5a_2022_2025.map(temizle_nan)
df_zmm059_2022_2025 = df_zmm059_2022_2025.map(temizle_nan)
df_me2n_2022_2025 = df_me2n_2022_2025.map(temizle_nan)
df_tedarikci_listesi = df_tedarikci_listesi.map(temizle_nan)
birlesik_df_zmb51 = birlesik_df_zmb51.map(temizle_nan)
df_zmm067_2022_2025 = df_zmm067_2022_2025.map(temizle_nan)
'''

'\ndf_me5a_2022_2025 = df_me5a_2022_2025.map(temizle_nan)\ndf_zmm059_2022_2025 = df_zmm059_2022_2025.map(temizle_nan)\ndf_me2n_2022_2025 = df_me2n_2022_2025.map(temizle_nan)\ndf_tedarikci_listesi = df_tedarikci_listesi.map(temizle_nan)\nbirlesik_df_zmb51 = birlesik_df_zmb51.map(temizle_nan)\ndf_zmm067_2022_2025 = df_zmm067_2022_2025.map(temizle_nan)\n'

In [28]:
df_calisilan = df_zmm059_2022_2025.copy()

In [29]:
df_calisilan["B3-B3POZ"] = (
    df_calisilan["Заявка"].astype(str) + "_" + df_calisilan["Позиция заявки"].astype(str)
)
df_me5a_2022_2025["B3-B3POZ"] = (
    df_me5a_2022_2025["Заявка"].astype(str) + "_" + df_me5a_2022_2025["Позиция заявки"].astype(str)
)

In [30]:
df_calisilan.columns.values[2] = "Зп_Поз"
df_tedarikci_listesi.rename(columns={'kod': 'Поставщик'}, inplace=True)

In [31]:
adetler = df_calisilan["B3-B3POZ"].value_counts()

In [32]:
mukerrer_kodlar = adetler[adetler > 1].index

In [33]:
silinmesi_gerekenler = df_calisilan[
    (df_calisilan["B3-B3POZ"].isin(mukerrer_kodlar)) &
    (df_calisilan["Документ_закупки"].isna())

]

In [34]:
df_calisilan = df_calisilan.drop(silinmesi_gerekenler.index)

In [35]:
#df_calisilan["Поставщик"] = df_calisilan["Поставщик"].apply(temizle_ve_cevir)

In [36]:
# Tedarikçi sözlüğünü yine string anahtarlara göre oluştur
tedarikci_dict = dict(zip(df_tedarikci_listesi.iloc[:, 0].astype(str), df_tedarikci_listesi.iloc[:, 1]))

# Eşleştirme
df_calisilan["имя_поставщика"] = df_calisilan["Поставщик"].map(tedarikci_dict)

In [37]:
#df_calisilan.tail()

In [38]:
# Sözlük oluştur (anahtar: B3-B3POZ, değer: Материал)
b3_to_material_dict = dict(zip(df_me5a_2022_2025["B3-B3POZ"].astype(str), df_me5a_2022_2025["Материал"]))

In [39]:
# B3-B3POZ'e göre eşleşen Материал'ları getir (yeni bir sütun olarak)
df_calisilan["Материал_yeni"] = df_calisilan["B3-B3POZ"].astype(str).map(b3_to_material_dict)

In [40]:
# Sadece boş olan Материал hücrelerini, eşleşen değerle doldur
df_calisilan["Материал"] = df_calisilan["Материал"].fillna(df_calisilan["Материал_yeni"])
# Geçici sütunu silelim
df_calisilan.drop(columns=["Материал_yeni"], inplace=True)

In [41]:
# Sözlükleri oluştur
ru_dict = dict(zip(df_material_list.iloc[:, 0].astype(str), df_material_list.iloc[:, 1]))
tr_dict = dict(zip(df_material_list.iloc[:, 0].astype(str), df_material_list.iloc[:, 2]))

# Eşleştir
df_calisilan["RU Наименование"] = df_calisilan["Материал"].astype(str).map(ru_dict)
df_calisilan["TR Наименование"] = df_calisilan["Материал"].astype(str).map(tr_dict)


In [42]:
#df_calisilan["Материал"].isna().sum()

In [43]:
#df_calisilan[["Поставщик", "имя_поставщика"]].sample(n=5)

In [44]:
df_calisilan.drop(columns=['Материалы тур.', 'Материалы анг.'], inplace=True)

In [45]:
# Sütunları çıkart
col_ru = df_calisilan.pop('RU Наименование')
col_tr = df_calisilan.pop('TR Наименование')
col_b3 = df_calisilan.pop('B3-B3POZ')

# Sütunları istediğin yerlere ekle
df_calisilan.insert(1, 'B3-B3POZ', col_b3)
df_calisilan.insert(8, 'RU Наименование', col_ru)
df_calisilan.insert(9, 'TR Наименование', col_tr)


In [46]:
 # Sözlük oluştur (anahtar: B3-B3POZ, değer: Материал)
b3_to_material_dict = dict(zip(df_me5a_2022_2025["B3-B3POZ"].astype(str), df_me5a_2022_2025["Материал"]))

In [47]:
# B3-B3POZ'e göre eşleşen Материал'ları getir (yeni bir sütun olarak)
df_calisilan["Материал_yeni"] = df_calisilan["B3-B3POZ"].astype(str).map(b3_to_material_dict)

In [48]:
# Sadece boş olan Материал hücrelerini, eşleşen değerle doldur
df_calisilan["Материал"] = df_calisilan["Материал"].fillna(df_calisilan["Материал_yeni"])
# Geçici sütunu silelim
df_calisilan.drop(columns=["Материал_yeni"], inplace=True)

In [49]:
# Sözlükleri oluştur
ru_dict = dict(zip(df_material_list.iloc[:, 0].astype(str), df_material_list.iloc[:, 1]))
tr_dict = dict(zip(df_material_list.iloc[:, 0].astype(str), df_material_list.iloc[:, 2]))

# Eşleştir
df_calisilan["RU Наименование"] = df_calisilan["Материал"].astype(str).map(ru_dict)
df_calisilan["TR Наименование"] = df_calisilan["Материал"].astype(str).map(tr_dict)

In [50]:
df_calisilan["Материал"].isna().sum()

np.int64(0)

In [51]:
df_calisilan = df_calisilan.map(temizle_nan)

In [52]:
sorted_columns = sorted(df_calisilan.columns)
print(sorted_columns)

['B3-B3POZ', 'Inventory number', 'PR / GR (%)', 'PR / PO (%)', 'RU Наименование', 'Requester', 'TR Наименование', 'Автор Внутреннего заказа', 'Автор ЗП в 1С', 'БЕ', 'Валюта', 'Валюта ЗП', 'Валюта.1', 'Валюта.2', 'Вид докум. закупки', 'Вид докум. закупки.1', 'Вид документа', 'Вид материала', 'Внутр. заказ заводу-изготовителю', 'ГОД', 'Группа закупок', 'Группа закупок.1', 'Группа закупок.2', 'Группа материалов', 'Групповой номер', 'Дата вступления в силу контракта', 'Дата доставки во Внутреннем заказ', 'Дата доставки, одобренная Поставщиком', 'Дата заявки', 'Дата контракта', 'Дата окончания действия договора', 'Дата поставки', 'Дата проводки', 'Дата создания', 'Дата создания.1', 'ДоговорнКоличество', 'Договорная цена нетто', 'Документ_закупки', 'Долгосрочный договор', 'ЕИ заказа на постав.', 'Единица ЗП', 'Единица измерения', 'Единица цены', 'Завод', 'Задержка', 'Закуп. организация', 'Запрос', 'Затребовал', 'Заявка', 'Значок "поставка завершена"', 'Зп_Поз', 'Имя 1', 'Имя 1.1', 'Имя клиен

In [53]:
#df_calisilan[["Документ_закупки","Зп_Поз","Количество ЗП","Цена нетто ЗП","Итого цена нетто"]].sample(n=5, random_state=1)

In [54]:
# Hepsini string yap ve boşlukları temizle
birlesik_df_zmb51["Заказ на поставку"] = birlesik_df_zmb51["Заказ на поставку"].astype(str).str.strip()
df_me2n_2022_2025["Документ закупки"] = df_me2n_2022_2025["Документ закупки"].astype(str).str.strip()


In [55]:
# "Наш знак" için dict
dict_nash_znak = dict(zip(df_me2n_2022_2025["Документ закупки"], df_me2n_2022_2025["Наш знак"]))

# "Ваш код" için dict
dict_vash_kod = dict(zip(df_me2n_2022_2025["Документ закупки"], df_me2n_2022_2025["Ваш код"]))

# map ile ekleme
birlesik_df_zmb51["Наш знак"] = birlesik_df_zmb51["Заказ на поставку"].map(dict_nash_znak)
birlesik_df_zmb51["Ваш код"] = birlesik_df_zmb51["Заказ на поставку"].map(dict_vash_kod)

In [56]:
df_calisilan["Позиция заявки"].apply(temizle_ve_cevir)

,Позиция заявки
0,80
1,90
2,100
3,70
4,60
...,...
816606,<NA>
816607,<NA>
816608,<NA>
816609,<NA>


In [57]:
# 1. Anahtar sütunlarını birleştir ve zp_zppoz adını ver
birlesik_df_zmb51["zp_zppoz"] = (
    birlesik_df_zmb51["Заказ на поставку"].apply(temizle_ve_cevir) + "_" +
    birlesik_df_zmb51["Позиция"].apply(temizle_ve_cevir)
)

df_calisilan["zp_zppoz"] = (
    df_calisilan["Документ_закупки"].apply(temizle_ve_cevir) + "_" +
    df_calisilan["Зп_Поз"].apply(temizle_ve_cevir)
)

# 2. dict oluştur
dict_poz = dict(zip(df_calisilan["zp_zppoz"], df_calisilan["Позиция заявки"]))

# 3. map ile yeni sütunu ekle
birlesik_df_zmb51["Позиция заявки"] = birlesik_df_zmb51["zp_zppoz"].map(dict_poz)
birlesik_df_zmb51["Позиция заявки"].apply(temizle_ve_cevir)
birlesik_df_zmb51["Позиция заявки"] = birlesik_df_zmb51["Позиция заявки"].fillna(0).astype(int)

/tmp/ipykernel_44620/873327287.py:18: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  birlesik_df_zmb51["Позиция заявки"] = birlesik_df_zmb51["Позиция заявки"].fillna(0).astype(int)


In [58]:
# 1. Zorunlu temizleme işlemleri (orijinal sütunların üzerine yazar)
df_me5a_2022_2025["Заявка"] = df_me5a_2022_2025["Заявка"].apply(temizle_ve_cevir)
df_me5a_2022_2025["Позиция заявки"] = df_me5a_2022_2025["Позиция заявки"].apply(temizle_ve_cevir)

df_zmm067_2022_2025["Заявка"] = df_zmm067_2022_2025["Заявка"].apply(temizle_ve_cevir)
df_zmm067_2022_2025["Позиция заявки"] = df_zmm067_2022_2025["Позиция заявки"].apply(temizle_ve_cevir)

# 2. Anahtar oluştur
df_me5a_2022_2025["zayavka_poz"] = df_me5a_2022_2025["Заявка"] + "_" + df_me5a_2022_2025["Позиция заявки"]
df_zmm067_2022_2025["zayavka_poz"] = df_zmm067_2022_2025["Заявка"] + "_" + df_zmm067_2022_2025["Позиция заявки"]

# 3. Dict oluştur
dict_dop = dict(zip(df_zmm067_2022_2025["zayavka_poz"], df_zmm067_2022_2025["Дополнительное"]))

# 4. map ile yeni sütun ekle
df_me5a_2022_2025["Дополнительное"] = df_me5a_2022_2025["zayavka_poz"].map(dict_dop)

#yeni oluşturulan sutunlara ihtiyaç kalmadı

df_me5a_2022_2025 = df_me5a_2022_2025.drop(columns=["zayavka_poz"])
df_zmm067_2022_2025 = df_zmm067_2022_2025.drop(columns=["zayavka_poz"])

In [59]:
# 1. Sütunları temizle (aynı sütunların üzerine yazıyoruz)
df_me2n_2022_2025["Документ закупки"] = df_me2n_2022_2025["Документ закупки"].apply(temizle_ve_cevir)
df_me2n_2022_2025["Позиция"] = df_me2n_2022_2025["Позиция"].apply(temizle_ve_cevir)

df_calisilan["Документ_закупки"] = df_calisilan["Документ_закупки"].apply(temizle_ve_cevir)
df_calisilan["Зп_Поз"] = df_calisilan["Зп_Поз"].apply(temizle_ve_cevir)

# 2. Anahtar oluştur
df_me2n_2022_2025["doc_poz"] = df_me2n_2022_2025["Документ закупки"] + "_" + df_me2n_2022_2025["Позиция"]
df_calisilan["doc_poz"] = df_calisilan["Документ_закупки"] + "_" + df_calisilan["Зп_Поз"]

# 3. Dict oluştur (anahtar: doc_poz, değer: Создал)
dict_sozdal = dict(zip(df_calisilan["doc_poz"], df_calisilan["Создал"]))

# 4. map ile df_me2n'e ekle
df_me2n_2022_2025["Создал"] = df_me2n_2022_2025["doc_poz"].map(dict_sozdal)

# yeni oluşturulan sütunlara ihtiyaç kalmadı
df_me2n_2022_2025 = df_me2n_2022_2025.drop(columns=["doc_poz"])
df_calisilan = df_calisilan.drop(columns=["doc_poz"])

In [60]:
# 1. Temizleme işlemi
birlesik_df_zmb51["Поставщик"] = birlesik_df_zmb51["Поставщик"].apply(temizle_ve_cevir)
df_tedarikci_listesi["﻿Кредитор"] = df_tedarikci_listesi["﻿Кредитор"].apply(temizle_ve_cevir)

# 2. Anahtar oluştur (tek sütun olduğu için direkt kullanabiliriz)
# 3. Dict oluştur
dict_suppliers = dict(zip(df_tedarikci_listesi["﻿Кредитор"], df_tedarikci_listesi["Имя поставщика"]))

# 4. map ile birlesik_df_zmb51'e ekle
birlesik_df_zmb51["Имя поставщика"] = birlesik_df_zmb51["Поставщик"].map(dict_suppliers)


In [61]:
# 1. Temizleme işlemi
birlesik_df_zmb51["Заказ на поставку"] = birlesik_df_zmb51["Заказ на поставку"].apply(temizle_ve_cevir)
birlesik_df_zmb51["Позиция"] = birlesik_df_zmb51["Позиция"].apply(temizle_ve_cevir)

df_me2n_2022_2025["Документ закупки"] = df_me2n_2022_2025["Документ закупки"].apply(temizle_ve_cevir)
df_me2n_2022_2025["Позиция"] = df_me2n_2022_2025["Позиция"].apply(temizle_ve_cevir)

# 2. Anahtar oluştur
birlesik_df_zmb51["doc_poz"] = birlesik_df_zmb51["Заказ на поставку"] + "_" + birlesik_df_zmb51["Позиция"]
df_me2n_2022_2025["doc_poz"] = df_me2n_2022_2025["Документ закупки"] + "_" + df_me2n_2022_2025["Позиция"]

# 3. Dict oluştur (anahtar: doc_poz, değer: Объем заказа)
dict_obem = dict(zip(df_me2n_2022_2025["doc_poz"], df_me2n_2022_2025["Объем заказа"]))

# 4. map ile birlesik_df_zmb51'e ekle
birlesik_df_zmb51["Объем заказа"] = birlesik_df_zmb51["doc_poz"].map(dict_obem)
#yeni oluşturulan sutunlara ihtiyaç kalmadı
birlesik_df_zmb51 = birlesik_df_zmb51.drop(columns=["doc_poz"])
df_me2n_2022_2025 = df_me2n_2022_2025.drop(columns=["doc_poz"])



In [62]:
df_me2n_2022_2025["Объем заказа"] = pd.to_numeric(df_me2n_2022_2025["Объем заказа"], errors="coerce")
df_me2n_2022_2025["еще поставить (количество)"] = pd.to_numeric(
    df_me2n_2022_2025["еще поставить (количество)"].fillna(0), errors="coerce"
)

# 2. Hesaplama
df_me2n_2022_2025["Уже поставлено (количество)"] = (
    df_me2n_2022_2025["Объем заказа"] - df_me2n_2022_2025["еще поставить (количество)"]
)

In [63]:
# 1. Temizleme işlemi
df_me2n_2022_2025["Документ закупки"] = df_me2n_2022_2025["Документ закупки"].apply(temizle_ve_cevir)
df_me2n_2022_2025["Позиция"] = df_me2n_2022_2025["Позиция"].apply(temizle_ve_cevir)


df_calisilan["Документ_закупки"] = df_calisilan["Документ_закупки"].apply(temizle_ve_cevir)
df_calisilan["Зп_Поз"] = df_calisilan["Зп_Поз"].apply(temizle_ve_cevir)

# 2. Anahtar oluştur
df_me2n_2022_2025["doc_poz"] = df_me2n_2022_2025["Документ закупки"] + "_" + df_me2n_2022_2025["Позиция"]
df_calisilan["doc_poz"] = df_calisilan["Документ_закупки"] + "_" + df_calisilan["Зп_Поз"]

# 3. Dict oluştur (anahtar: doc_poz, değer: Позиция заявки)
dict_poz = dict(zip(df_calisilan["doc_poz"], df_calisilan["Позиция заявки"]))

# 4. map ile df_me2n_2022_2025'e ekle
df_me2n_2022_2025["Позиция заявки"] = df_me2n_2022_2025["doc_poz"].map(dict_poz)
#yeni oluşturulan sutunlara ihtiyaç kalmadı
df_calisilan = df_calisilan.drop(columns=["doc_poz"])
df_me2n_2022_2025 = df_me2n_2022_2025.drop(columns=["doc_poz"])


In [64]:
df_me2n_2022_2025["Материал_ME2N"] = df_me2n_2022_2025["Материал"]
df_me5a_2022_2025["Материал_ME5A"] = df_me5a_2022_2025["Материал"]
# 1. Temizleme işlemi
df_me2n_2022_2025["Заявка"] = df_me2n_2022_2025["Заявка"].apply(temizle_ve_cevir)
df_me2n_2022_2025["Позиция заявки"] = df_me2n_2022_2025["Позиция заявки"].apply(temizle_ve_cevir)

df_me5a_2022_2025["Заявка"] = df_me5a_2022_2025["Заявка"].apply(temizle_ve_cevir)
df_me5a_2022_2025["Позиция заявки"] = df_me5a_2022_2025["Позиция заявки"].apply(temizle_ve_cevir)

# 2. Anahtar oluştur
df_me2n_2022_2025["zayavka_poz"] = df_me2n_2022_2025["Заявка"] + "_" + df_me2n_2022_2025["Позиция заявки"]
df_me5a_2022_2025["zayavka_poz"] = df_me5a_2022_2025["Заявка"] + "_" + df_me5a_2022_2025["Позиция заявки"]

# 3. Dict oluştur (anahtar: zayavka_poz, değer: Материал_ME5A)
dict_me5a = dict(zip(df_me5a_2022_2025["zayavka_poz"], df_me5a_2022_2025["Материал_ME5A"]))

# 4. map ile df_me2n_2022_2025'e ekle
df_me2n_2022_2025["Материал_ME5A"] = df_me2n_2022_2025["zayavka_poz"].map(dict_me5a)
#yeni oluşturulan sutunlara ihtiyaç kalmadı
df_me5a_2022_2025 = df_me5a_2022_2025.drop(columns=["zayavka_poz"])
df_me2n_2022_2025 = df_me2n_2022_2025.drop(columns=["zayavka_poz"])


In [65]:
df_me2n_2022_2025 = df_me2n_2022_2025.applymap(temizle_nan)

/tmp/ipykernel_44620/2153232079.py:1: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_me2n_2022_2025 = df_me2n_2022_2025.applymap(temizle_nan)


In [66]:
df_me2n_2022_2025_analog = df_me2n_2022_2025[
    (df_me2n_2022_2025["Материал_ME2N"] != df_me2n_2022_2025["Материал_ME5A"]) &
    (df_me2n_2022_2025["Заявка"].apply(dolu_mi) & df_me2n_2022_2025["Позиция заявки"].apply(dolu_mi))
].copy()


In [67]:
# 1. Temizleme işlemi (gerekiyorsa)
df_me2n_2022_2025_analog["Материал_ME5A"] = df_me2n_2022_2025_analog["Материал_ME5A"].apply(temizle_ve_cevir)
df_material_list["﻿Материал"] = df_material_list["﻿Материал"].apply(temizle_ve_cevir)

# 2. Dict oluştur
dict_tr = dict(zip(df_material_list["﻿Материал"], df_material_list["TR Наименование"]))
dict_ru = dict(zip(df_material_list["﻿Материал"], df_material_list["RU Наименование"]))

# 3. map ile yeni sütunları ekle
df_me2n_2022_2025_analog["TR Наименование_ВЗ"] = df_me2n_2022_2025_analog["Материал_ME5A"].map(dict_tr)
df_me2n_2022_2025_analog["RU Наименование_ВЗ"] = df_me2n_2022_2025_analog["Материал_ME5A"].map(dict_ru)


In [68]:
df_me2n_2022_2025_analog = df_me2n_2022_2025_analog.rename(columns={"Наименование RU": "Наименование RU_ЗП"})
df_me2n_2022_2025_analog = df_me2n_2022_2025_analog.rename(columns={"Наименование TR": "Наименование TR_ЗП"})


In [69]:
# 1. Anahtar sütunları tuple haline getirelim
df_me2n_2022_2025["key"] = df_me2n_2022_2025[["Документ закупки", "Позиция"]].apply(tuple, axis=1)
df_me5a_2022_2025["key"] = df_me5a_2022_2025[["Заказ на поставку", "Позиция ЗкзНаПостав"]].apply(tuple, axis=1)

# 2. Aktarılacak sütunlar ve hedef adları
map_dict = {
    "Объем заказа": "Объем заказа",
    "еще поставить (количество)": "еще поставить (количество)",
    "Еще для поставки (стоимость)": "Еще для поставки (стоимость)",
    "Для фактурирования (колич.)": "Для фактурирования (колич.)",
    "Для фактурирования (стоим.)": "Для фактурирования (стоим.)",
    "Имя поставщика": "Имя поставщика",
    "Создал": "ЗП_Создал",
    "Наш знак":"Наш знак",
    "Цена нетто":"Цена нетто",
    "Валюта":"Валюта",
    "Индикатор удаления":"Индикатор удаления_зп",
    "ЕИ заказа на постав.":"ЕИ заказа на постав.",
    "Единица цены":"Единица цены",

}

# 3. Her sütun için dict map uygulama
for source_col, target_col in map_dict.items():
    temp_dict = dict(zip(df_me2n_2022_2025["key"], df_me2n_2022_2025[source_col]))
    df_me5a_2022_2025[target_col] = df_me5a_2022_2025["key"].map(temp_dict)

# 4. Geçici key sütunlarını kaldır
df_me2n_2022_2025.drop(columns=["key"], inplace=True)
df_me5a_2022_2025.drop(columns=["key"], inplace=True)


In [70]:
# 1. Yeni sütun sırasını belirle
df_me2n_2022_2025_analog_yeni_sira = [
    'ГОД', 'Документ закупки', 'Позиция', 'Группа закупок',
    'Закуп. организация','Создал', 'Индикатор удаления', 'Дата документа',
    'Наш знак', 'Ваш код', 'Имя поставщика', 'Заявка',  'Позиция заявки',
    'Материал_ME2N','Материал_ME5A','Наименование RU_ЗП','Наименование TR_ЗП',
    'RU Наименование_ВЗ','TR Наименование_ВЗ',
    'Объем заказа', 'еще поставить (количество)','Уже поставлено (количество)',
    'Складская ЕИ','Количество в СЕИ','Единица цены','Цена нетто',
    'СтоимЗаказа нетто','Еще для поставки (стоимость)','Валюта',
    'Вид докум. закупки','Код налога','Дата поставки','ЕИ заказа на постав.',
    'Инвентарный номер'
]

# 2. DataFrame'i yeni sütun sırasına göre düzenle
df_me2n_2022_2025_analog =df_me2n_2022_2025_analog[df_me2n_2022_2025_analog_yeni_sira]


In [71]:
# 1. Yeni sütun sırasını belirle
df_me2n_2022_2025_yeni_sira = [
      'ГОД', 'Документ закупки', 'Позиция','Создал', 'Группа закупок',
      'Закуп. организация', 'Индикатор удаления', 'Дата документа',
      'Наш знак', 'Ваш код', 'Имя поставщика', 'Заявка','Позиция заявки',
      'Материал','Инвентарный номер', 'Наименование RU','Наименование TR',
      'Объем заказа','Складская ЕИ','еще поставить (количество)',
      'Еще для поставки (стоимость)','Уже поставлено (количество)',
      'Для фактурирования (стоим.)','Для фактурирования (колич.)',
      'Количество в СЕИ', 'Единица цены','СтоимЗаказа нетто','Валюта',
      'Цена нетто','Вид докум. закупки', 'Код налога','Краткий текст',
      'Дата поставки', 'ЕИ заказа на постав.','ТипДокумЗакупки',
      'Тип неполноты данных','Группа материалов','Материал_ME2N', 'Материал_ME5A'

]

# 2. DataFrame'i yeni sütun sırasına göre düzenle
df_me2n_2022_2025 =df_me2n_2022_2025[df_me2n_2022_2025_yeni_sira]

In [72]:
df_me5a_2022_2025.rename(columns={"'Дата заявки": "Дата_заявки"}, inplace=True)


In [73]:
df_me5a_2022_2025_yeni_sira = [
       'ГОД', 'Заявка', 'Позиция заявки', 'Материал', 'Индикатор удаления_B3','ЗатребованКолич',
       'Объем заказа',  'Заказ на поставку', 'Позиция ЗкзНаПостав','Индикатор удаления_зп',
       'Затребовал', 'Индик. выдачи', 'Группа деблокир.',
       'Статус обработки заявки', 'Дата заявки', 'Группа закупок',
       'Дата изменения','Дата заказа', 'Долгосрочный договор',
       'Заказанное к-во', 'Единица измерения', 'RU Наименование',
       'TR Наименование',  'Дополнительное','Инвентарный номер', 'Краткий текст',
       'Закуп. организация','еще поставить (количество)','Еще для поставки (стоимость)',
       'Для фактурирования (колич.)','Для фактурирования (стоим.)', 'Имя поставщика',
       'ЗП_Создал','Наш знак', 'Цена нетто', 'Валюта', 'ЕИ заказа на постав.',
       'Единица цены', 'Создал', 'ПоступлМатериала', 'Поступление счета',
       'Номер проекта', 'Группа материалов', 'Дата поставки',
       'Завод-поставщик', 'Вид документа', 'Статус обработки',
       'Индикатор создания', 'Стратегия деблокир.', 'НДП-материал',
       'Номер потребности', 'Объем дефицита', 'Тип даты поставки',
       'Дата деблокирования', 'Дата_заявки', 'B3-B3POZ','Материал_ME5A'

]

# 2. DataFrame'i yeni sütun sırasına göre düzenle
df_me5a_2022_2025 =df_me5a_2022_2025[df_me5a_2022_2025_yeni_sira]

In [74]:
df_calisilan_yeni_sira = [
    'ГОД','zp_zppoz','B3-B3POZ','Документ_закупки', 'Зп_Поз',
    'Индикатор удаления','неполн.', 'Материал', 'Заявка', 'Позиция заявки',
    'RU Наименование', 'TR Наименование',
    'Инвентарный номер в SAS', 'Количество ЗП',
    'Минимальная цена ЗП', 'Минимальная цена ЗП.1',
    'Минимальная цена ЗП.2','Единица ЗП', 'Цена нетто ЗП', 'Итого цена нетто',
    'Валюта ЗП','Вид докум. закупки', 'Дата создания', 'Дата поставки',
    'Дата доставки, одобренная Поставщиком', 'Группа закупок',
    'Название ГрЗакупок', 'БЕ', 'Название фирмы', 'Закуп. организация',
    'Название ЗакупОрг', 'Завод', 'Имя 1', 'Поставщик', 'Группа материалов',
    'Название группы', 'Краткий текст', 'Создал', 'Вид докум. закупки.1',
    'Обозначение ВидДокум', 'Дата создания.1', 'Создал.1','Группа закупок.1',
    'Название ГрЗакупок.1', 'ЕИ заказа на постав.','Единица цены',
    'Цена заказа нетто', 'Стоимость брутто', 'Валюта','Недопоставленное количество',
    'Количество поставки', 'Сумма брутто','Валюта.1', 'Количество по счету', 'Дата проводки',
    'Количество по счету.1', 'Валюта.2', 'Остаток количества по счету',
    'Остаток суммы по счету', 'Автор Внутреннего заказа', 'Количество по счету.2',
    'Значок "поставка завершена"','Дата контракта', 'Дата вступления в силу контракта',
    'Дата окончания действия договора', 'Склад', '№ комиссии','имя_поставщика'
]

# 2. DataFrame'i yeni sütun sırasına göre düzenle
df_calisilan =df_calisilan[df_calisilan_yeni_sira]

In [75]:
# 1. Anahtar sütunları tuple haline getirelim
birlesik_df_zmb51["key"] = birlesik_df_zmb51[["Материал"]].apply(tuple, axis=1)
df_material_list["key"] = df_material_list[["﻿Материал"]].apply(tuple, axis=1)

# 2. Aktarılacak sütunlar ve hedef adları
map_dict_B3_status = {
    "Группа материалов":"Группа материалов",

}

# 3. Her sütun için dict map uygulama
for source_col, target_col in map_dict_B3_status.items():
    temp_dict = dict(zip(df_material_list["key"], df_material_list[source_col]))
    birlesik_df_zmb51[target_col] = birlesik_df_zmb51["key"].map(temp_dict)

# 4. Geçici key sütunlarını kaldır
birlesik_df_zmb51.drop(columns=["key"], inplace=True)
df_material_list.drop(columns=["key"], inplace=True)

In [76]:
birlesik_df_zmb51.columns

Index(['ГОД', 'Вид движения', 'Партия', 'Материал', 'Склад', 'Дата ввода',
       'Время ввода', 'Дата документа', 'Дата проводки', 'Внутренний заказ',
       'Заказ на поставку', 'Позиция', 'Документ материала',
       'Поз. док. материала', 'Признак 2', 'Признак 5', 'Ссылка',
       'Краткий текст материала', 'Кол-во в ЕИ ввода', 'ЕИ ввода',
       'Сумма во ВВ', 'Количество', 'Поставщик', 'Накладная',
       'Текст заголовка документа', 'Заявитель', 'Инвентарный номер',
       'Имя пользователя', 'БЕ', 'Базовая ЕИ', 'ГодДокумМатериала', 'Валюта',
       'Завод', 'Имя 1', 'Основное средство', 'ЕИ цены заказа',
       'ЕИ заказа на постав.', 'Наш знак', 'Ваш код', 'zp_zppoz',
       'Позиция заявки', 'Имя поставщика', 'Объем заказа',
       'Группа материалов'],
      dtype='object')

In [77]:
# 1. Yeni sütun sırasını belirle
zmb51_yeni_sira = [
    'ГОД', 'Вид движения', 'Партия', 'Материал', 'Группа материалов','Заказ на поставку',
    'Позиция', 'Наш знак', 'Ваш код', 'Внутренний заказ', 'Позиция заявки',
    'Склад', 'Дата ввода', 'Время ввода', 'Дата документа', 'Дата проводки',
    'Документ материала', 'Поз. док. материала', 'Признак 2','Признак 5', 'Ссылка',
    'Краткий текст материала', 'Кол-во в ЕИ ввода', 'Объем заказа',
    'ЕИ ввода','Сумма во ВВ', 'Количество', 'Поставщик','Имя поставщика',
    'Накладная','Текст заголовка документа', 'Заявитель', 'Инвентарный номер',
    'Имя пользователя', 'БЕ', 'Базовая ЕИ', 'ГодДокумМатериала', 'Валюта',
    'Завод', 'Имя 1', 'Основное средство', 'ЕИ цены заказа','ЕИ заказа на постав.'

]

# 2. DataFrame'i yeni sütun sırasına göre düzenle
birlesik_df_zmb51 = birlesik_df_zmb51[zmb51_yeni_sira]

In [78]:
#ME5A
df_me5a_2022_2025["ГОД"] = pd.to_numeric(df_me5a_2022_2025["ГОД"], errors="coerce").astype("Int64")
df_me5a_2022_2025["Заявка"] = pd.to_numeric(df_me5a_2022_2025["Заявка"], errors="coerce").astype("Int64")
df_me5a_2022_2025["Материал"] = pd.to_numeric(df_me5a_2022_2025["Материал"], errors="coerce").astype("Int64")
df_me5a_2022_2025["Заказ на поставку"] = pd.to_numeric(df_me5a_2022_2025["Заказ на поставку"], errors="coerce").astype("Int64")
df_me5a_2022_2025["Позиция ЗкзНаПостав"] = pd.to_numeric(df_me5a_2022_2025["Позиция ЗкзНаПостав"], errors="coerce").astype("Int64")
df_me5a_2022_2025["ЗатребованКолич"] = pd.to_numeric(df_me5a_2022_2025["ЗатребованКолич"], errors="coerce").astype("float64")
df_me5a_2022_2025["Заказанное к-во"] = pd.to_numeric(df_me5a_2022_2025["Заказанное к-во"], errors="coerce").astype("float64")
df_me5a_2022_2025["Объем заказа"] = pd.to_numeric(df_me5a_2022_2025["Объем заказа"], errors="coerce").astype("float64")
df_me5a_2022_2025["еще поставить (количество)"] = pd.to_numeric(df_me5a_2022_2025["еще поставить (количество)"], errors="coerce").astype("float64")
df_me5a_2022_2025["Еще для поставки (стоимость)"] = pd.to_numeric(df_me5a_2022_2025["Еще для поставки (стоимость)"], errors="coerce").astype("float64")
df_me5a_2022_2025["Для фактурирования (колич.)"] = pd.to_numeric(df_me5a_2022_2025["Для фактурирования (колич.)"], errors="coerce").astype("float64")
df_me5a_2022_2025["Для фактурирования (стоим.)"] = pd.to_numeric(df_me5a_2022_2025["Для фактурирования (стоим.)"], errors="coerce").astype("float64")
#ME2N
df_me2n_2022_2025["ГОД"] = pd.to_numeric(df_me2n_2022_2025["ГОД"], errors="coerce").astype("Int64")
df_me2n_2022_2025["Документ закупки"] = pd.to_numeric(df_me2n_2022_2025["Документ закупки"], errors="coerce").astype("Int64")
df_me2n_2022_2025["Позиция"] = pd.to_numeric(df_me2n_2022_2025["Позиция"], errors="coerce").astype("Int64")
df_me2n_2022_2025["Заявка"] = pd.to_numeric(df_me2n_2022_2025["Заявка"], errors="coerce").astype("Int64")
df_me2n_2022_2025["Материал"] = pd.to_numeric(df_me2n_2022_2025["Материал"], errors="coerce").astype("Int64")
df_me2n_2022_2025["Объем заказа"] = pd.to_numeric(df_me2n_2022_2025["Объем заказа"], errors="coerce").astype("float64")
df_me2n_2022_2025["еще поставить (количество)"] = pd.to_numeric(df_me2n_2022_2025["еще поставить (количество)"], errors="coerce").astype("float64")
df_me2n_2022_2025["Уже поставлено (количество)"] = pd.to_numeric(df_me2n_2022_2025["Уже поставлено (количество)"], errors="coerce").astype("float64")
df_me2n_2022_2025["Для фактурирования (стоим.)"] = pd.to_numeric(df_me2n_2022_2025["Для фактурирования (стоим.)"], errors="coerce").astype("float64")
df_me2n_2022_2025["Для фактурирования (колич.)"] = pd.to_numeric(df_me2n_2022_2025["Для фактурирования (колич.)"], errors="coerce").astype("float64")
df_me2n_2022_2025["Количество в СЕИ"] = pd.to_numeric(df_me2n_2022_2025["Количество в СЕИ"], errors="coerce").astype("float64")
df_me2n_2022_2025["Цена нетто"] = pd.to_numeric(df_me2n_2022_2025["Цена нетто"], errors="coerce").astype("float64")
df_me2n_2022_2025["СтоимЗаказа нетто"] = pd.to_numeric(df_me2n_2022_2025["СтоимЗаказа нетто"], errors="coerce").astype("float64")
df_me2n_2022_2025["Еще для поставки (стоимость)"] = pd.to_numeric(df_me2n_2022_2025["Еще для поставки (стоимость)"], errors="coerce").astype("float64")
#zmb51
birlesik_df_zmb51["ГОД"] = pd.to_numeric(birlesik_df_zmb51["ГОД"], errors="coerce").astype("Int64")
birlesik_df_zmb51["Партия"] = pd.to_numeric(birlesik_df_zmb51["Партия"], errors="coerce").astype("Int64")
birlesik_df_zmb51["Заказ на поставку"] = pd.to_numeric(birlesik_df_zmb51["Заказ на поставку"], errors="coerce").astype("Int64")
birlesik_df_zmb51["Позиция"] = pd.to_numeric(birlesik_df_zmb51["Позиция"], errors="coerce").astype("Int64")
birlesik_df_zmb51["Внутренний заказ"] = pd.to_numeric(birlesik_df_zmb51["Внутренний заказ"], errors="coerce").astype("Int64")
birlesik_df_zmb51["Документ материала"] = pd.to_numeric(birlesik_df_zmb51["Документ материала"], errors="coerce").astype("Int64")
birlesik_df_zmb51["Поз. док. материала"] = pd.to_numeric(birlesik_df_zmb51["Поз. док. материала"], errors="coerce").astype("Int64")
birlesik_df_zmb51["Кол-во в ЕИ ввода"] = pd.to_numeric(birlesik_df_zmb51["Кол-во в ЕИ ввода"], errors="coerce").astype("float64")
birlesik_df_zmb51["Сумма во ВВ"] = pd.to_numeric(birlesik_df_zmb51["Сумма во ВВ"], errors="coerce").astype("float64")
birlesik_df_zmb51["Количество"] = pd.to_numeric(birlesik_df_zmb51["Количество"], errors="coerce").astype("float64")
birlesik_df_zmb51["Поставщик"] = pd.to_numeric(birlesik_df_zmb51["Поставщик"], errors="coerce").astype("Int64")
#zmm059
df_calisilan["ГОД"] = pd.to_numeric(df_calisilan["ГОД"], errors="coerce").astype("Int64")
df_calisilan["Документ_закупки"] = pd.to_numeric(df_calisilan["Документ_закупки"], errors="coerce").astype("Int64")
df_calisilan["Зп_Поз"] = pd.to_numeric(df_calisilan["Зп_Поз"], errors="coerce").astype("Int64")
df_calisilan["Материал"] = pd.to_numeric(df_calisilan["Материал"], errors="coerce").astype("Int64")
df_calisilan["Заявка"] = pd.to_numeric(df_calisilan["Заявка"], errors="coerce").astype("Int64")
df_calisilan["Позиция заявки"] = pd.to_numeric(df_calisilan["Позиция заявки"], errors="coerce").astype("Int64")
df_calisilan["Количество ЗП"] = pd.to_numeric(df_calisilan["Количество ЗП"], errors="coerce").astype("float64")
df_calisilan["Минимальная цена ЗП"] = pd.to_numeric(df_calisilan["Минимальная цена ЗП"], errors="coerce").astype("float64")
df_calisilan["Минимальная цена ЗП.1"] = pd.to_numeric(df_calisilan["Минимальная цена ЗП.1"], errors="coerce").astype("float64")
df_calisilan["Единица ЗП"] = pd.to_numeric(df_calisilan["Единица ЗП"], errors="coerce").astype("Int64")
df_calisilan["Цена нетто ЗП"] = pd.to_numeric(df_calisilan["Цена нетто ЗП"], errors="coerce").astype("float64")
df_calisilan["Итого цена нетто"] = pd.to_numeric(df_calisilan["Итого цена нетто"], errors="coerce").astype("float64")
df_calisilan["Поставщик"] = pd.to_numeric(df_calisilan["Поставщик"], errors="coerce").astype("Int64")
#me2n_analog
df_me2n_2022_2025_analog["ГОД"] = pd.to_numeric(df_me2n_2022_2025["ГОД"], errors="coerce").astype("Int64")
df_me2n_2022_2025_analog["Документ закупки"] = pd.to_numeric(df_me2n_2022_2025["Документ закупки"], errors="coerce").astype("Int64")
df_me2n_2022_2025_analog["Заявка"] = pd.to_numeric(df_me2n_2022_2025["Заявка"], errors="coerce").astype("Int64")
df_me2n_2022_2025_analog["Материал"] = pd.to_numeric(df_me2n_2022_2025["Материал"], errors="coerce").astype("Int64")
df_me2n_2022_2025_analog["Объем заказа"] = pd.to_numeric(df_me2n_2022_2025["Объем заказа"], errors="coerce").astype("float64")
df_me2n_2022_2025_analog["еще поставить (количество)"] = pd.to_numeric(df_me2n_2022_2025["еще поставить (количество)"], errors="coerce").astype("float64")
df_me2n_2022_2025_analog["Количество в СЕИ"] = pd.to_numeric(df_me2n_2022_2025["Количество в СЕИ"], errors="coerce").astype("float64")
df_me2n_2022_2025_analog["Цена нетто"] = pd.to_numeric(df_me2n_2022_2025["Цена нетто"], errors="coerce").astype("float64")
df_me2n_2022_2025_analog["СтоимЗаказа нетто"] = pd.to_numeric(df_me2n_2022_2025["СтоимЗаказа нетто"], errors="coerce").astype("float64")
df_me2n_2022_2025_analog["Еще для поставки (стоимость)"] = pd.to_numeric(df_me2n_2022_2025["Еще для поставки (стоимость)"], errors="coerce").astype("float64")

#zmm067
df_zmm067_2022_2025.rename(columns={"SAT Yaratma Tarihi": "Дата Заявка"}, inplace=True)
df_zmm067_2022_2025.rename(columns={"Sipariş Silme Göstergesi": "Индикатор удаления заказа"}, inplace=True)
df_zmm067_2022_2025.rename(columns={"SAT Teslimat Tarihi": "Дата доставки заявка"}, inplace=True)
df_zmm067_2022_2025.rename(columns={"Talep Silme Göstergesi": "Индикатор удаления заявка"}, inplace=True)
df_zmm067_2022_2025["Объем заказа"] = pd.to_numeric(df_zmm067_2022_2025["Объем заказа"], errors="coerce").astype("float64")

In [79]:
df_calisilan_cleaned =df_calisilan.copy()

In [80]:
df_me5a_cleaned.columns

Index(['ГОД', 'Заявка', 'Позиция заявки', 'Материал', 'Индикатор удаления_B3',
       'Затребовал', 'Индик. выдачи', 'Группа деблокир.',
       'Статус обработки заявки', 'Дата заявки', 'Группа закупок',
       'Дата изменения', 'Заказ на поставку', 'Позиция ЗкзНаПостав',
       'Дата заказа', 'Долгосрочный договор', 'ЗатребованКолич',
       'Заказанное к-во', 'Единица измерения', 'RU Наименование',
       'TR Наименование', 'Инвентарный номер', 'Краткий текст',
       'Закуп. организация', 'Создал', 'ПоступлМатериала', 'Поступление счета',
       'Номер проекта', 'Группа материалов', 'Дата поставки',
       'Завод-поставщик', 'Вид документа', 'Статус обработки',
       'Индикатор создания', 'Стратегия деблокир.', 'НДП-материал',
       'Номер потребности', 'Объем дефицита', 'Тип даты поставки',
       'Дата деблокирования', ''Дата заявки'],
      dtype='object')

In [81]:
df_me5a_cleaned = df_me5a_cleaned[~df_me5a_cleaned["Индикатор удаления_B3"]]

In [82]:
# sadece NaN olanları tutar; boş string ('') varsa bu satırlar silinebilir
df_calisilan_cleaned = df_calisilan_cleaned[df_calisilan_cleaned['Индикатор удаления'].isna()]


In [83]:
df_calisilan_cleaned['Индикатор удаления']

,Индикатор удаления
0,<NA>
1,<NA>
2,<NA>
3,<NA>
4,<NA>
...,...
816606,<NA>
816607,<NA>
816608,<NA>
816609,<NA>


In [84]:
df_calisilan_cleaned.columns


Index(['ГОД', 'zp_zppoz', 'B3-B3POZ', 'Документ_закупки', 'Зп_Поз',
       'Индикатор удаления', 'неполн.', 'Материал', 'Заявка', 'Позиция заявки',
       'RU Наименование', 'TR Наименование', 'Инвентарный номер в SAS',
       'Количество ЗП', 'Минимальная цена ЗП', 'Минимальная цена ЗП.1',
       'Минимальная цена ЗП.2', 'Единица ЗП', 'Цена нетто ЗП',
       'Итого цена нетто', 'Валюта ЗП', 'Вид докум. закупки', 'Дата создания',
       'Дата поставки', 'Дата доставки, одобренная Поставщиком',
       'Группа закупок', 'Название ГрЗакупок', 'БЕ', 'Название фирмы',
       'Закуп. организация', 'Название ЗакупОрг', 'Завод', 'Имя 1',
       'Поставщик', 'Группа материалов', 'Название группы', 'Краткий текст',
       'Создал', 'Вид докум. закупки.1', 'Обозначение ВидДокум',
       'Дата создания.1', 'Создал.1', 'Группа закупок.1',
       'Название ГрЗакупок.1', 'ЕИ заказа на постав.', 'Единица цены',
       'Цена заказа нетто', 'Стоимость брутто', 'Валюта',
       'Недопоставленное количест

In [85]:
df_me5a_cleaned.columns

Index(['ГОД', 'Заявка', 'Позиция заявки', 'Материал', 'Индикатор удаления_B3',
       'Затребовал', 'Индик. выдачи', 'Группа деблокир.',
       'Статус обработки заявки', 'Дата заявки', 'Группа закупок',
       'Дата изменения', 'Заказ на поставку', 'Позиция ЗкзНаПостав',
       'Дата заказа', 'Долгосрочный договор', 'ЗатребованКолич',
       'Заказанное к-во', 'Единица измерения', 'RU Наименование',
       'TR Наименование', 'Инвентарный номер', 'Краткий текст',
       'Закуп. организация', 'Создал', 'ПоступлМатериала', 'Поступление счета',
       'Номер проекта', 'Группа материалов', 'Дата поставки',
       'Завод-поставщик', 'Вид документа', 'Статус обработки',
       'Индикатор создания', 'Стратегия деблокир.', 'НДП-материал',
       'Номер потребности', 'Объем дефицита', 'Тип даты поставки',
       'Дата деблокирования', ''Дата заявки'],
      dtype='object')

In [86]:
df_me5a_cleaned.loc[:, 'zp_zppoz'] = (
    df_me5a_cleaned['Заказ на поставку'].astype(str) + '_' +
    df_me5a_cleaned['Позиция ЗкзНаПостав'].astype(str)
)

In [87]:
df_me5a_cleaned.loc[:, 'B3-POZ'] = (
    df_me5a_cleaned['Заявка'].astype(str) + '_' +
    df_me5a_cleaned['Позиция заявки'].astype(str)
)

In [88]:
df_me5a_cleaned.loc[df_me5a_cleaned['Заказ на поставку'].notna(), 'Заказ на поставку'] = pd.NA
df_me5a_cleaned.loc[df_me5a_cleaned['Позиция ЗкзНаПостав'].notna(), 'Позиция ЗкзНаПостав'] = pd.NA

In [89]:
df_me5a_cleaned['ЗатребованКолич']

,ЗатребованКолич
0,4
1,4
2,600
3,5
4,20
...,...
407108,2
407109,2
407110,30
407111,30


In [90]:
df_me5a_cleaned.loc[:, "Заказанное к-во"] = pd.to_numeric(
    df_me5a_cleaned["Заказанное к-во"], errors="coerce"
).astype("float64")

In [91]:
df_me5a_cleaned.loc[:, "ЗатребованКолич"] = pd.to_numeric(
    df_me5a_cleaned["ЗатребованКолич"], errors="coerce"
).astype("float64")

In [92]:
df_me5a_cleaned.columns

Index(['ГОД', 'Заявка', 'Позиция заявки', 'Материал', 'Индикатор удаления_B3',
       'Затребовал', 'Индик. выдачи', 'Группа деблокир.',
       'Статус обработки заявки', 'Дата заявки', 'Группа закупок',
       'Дата изменения', 'Заказ на поставку', 'Позиция ЗкзНаПостав',
       'Дата заказа', 'Долгосрочный договор', 'ЗатребованКолич',
       'Заказанное к-во', 'Единица измерения', 'RU Наименование',
       'TR Наименование', 'Инвентарный номер', 'Краткий текст',
       'Закуп. организация', 'Создал', 'ПоступлМатериала', 'Поступление счета',
       'Номер проекта', 'Группа материалов', 'Дата поставки',
       'Завод-поставщик', 'Вид документа', 'Статус обработки',
       'Индикатор создания', 'Стратегия деблокир.', 'НДП-материал',
       'Номер потребности', 'Объем дефицита', 'Тип даты поставки',
       'Дата деблокирования', ''Дата заявки', 'zp_zppoz', 'B3-POZ'],
      dtype='object')

In [93]:
df_calisilan_cleaned.columns

Index(['ГОД', 'zp_zppoz', 'B3-B3POZ', 'Документ_закупки', 'Зп_Поз',
       'Индикатор удаления', 'неполн.', 'Материал', 'Заявка', 'Позиция заявки',
       'RU Наименование', 'TR Наименование', 'Инвентарный номер в SAS',
       'Количество ЗП', 'Минимальная цена ЗП', 'Минимальная цена ЗП.1',
       'Минимальная цена ЗП.2', 'Единица ЗП', 'Цена нетто ЗП',
       'Итого цена нетто', 'Валюта ЗП', 'Вид докум. закупки', 'Дата создания',
       'Дата поставки', 'Дата доставки, одобренная Поставщиком',
       'Группа закупок', 'Название ГрЗакупок', 'БЕ', 'Название фирмы',
       'Закуп. организация', 'Название ЗакупОрг', 'Завод', 'Имя 1',
       'Поставщик', 'Группа материалов', 'Название группы', 'Краткий текст',
       'Создал', 'Вид докум. закупки.1', 'Обозначение ВидДокум',
       'Дата создания.1', 'Создал.1', 'Группа закупок.1',
       'Название ГрЗакупок.1', 'ЕИ заказа на постав.', 'Единица цены',
       'Цена заказа нетто', 'Стоимость брутто', 'Валюта',
       'Недопоставленное количест

In [94]:
# 1️⃣ Önce df_calisilan_cleaned'deki anahtarları benzersiz hale getiriyoruz
df_calisilan_unique = df_calisilan_cleaned.drop_duplicates(subset='B3-B3POZ')

# 2️⃣ Merge işlemini yapıyoruz (her iki sütunu da ekliyoruz)
df_me5a_cleaned = df_me5a_cleaned.merge(
    df_calisilan_unique[['B3-B3POZ', 'Документ_закупки', 'Зп_Поз']],
    left_on='B3-POZ',
    right_on='B3-B3POZ',
    how='left'
)

# 3️⃣ Dolularını kendi sütunlarına yazıyoruz
df_me5a_cleaned['Заказ на поставку'] = df_me5a_cleaned['Документ_закупки'].combine_first(
    df_me5a_cleaned['Заказ на поставку']
)
df_me5a_cleaned['Позиция ЗкзНаПостав'] = df_me5a_cleaned['Зп_Поз'].combine_first(
    df_me5a_cleaned['Позиция ЗкзНаПостав']
)

# 4️⃣ Geçici sütunları güvenli şekilde sil
cols_to_drop = [col for col in ['Документ_закупки', 'Зп_Поз', 'B3-B3POZ'] if col in df_me5a_cleaned.columns]
df_me5a_cleaned.drop(columns=cols_to_drop, inplace=True)


In [95]:
df_me5a_cleaned.columns

Index(['ГОД', 'Заявка', 'Позиция заявки', 'Материал', 'Индикатор удаления_B3',
       'Затребовал', 'Индик. выдачи', 'Группа деблокир.',
       'Статус обработки заявки', 'Дата заявки', 'Группа закупок',
       'Дата изменения', 'Заказ на поставку', 'Позиция ЗкзНаПостав',
       'Дата заказа', 'Долгосрочный договор', 'ЗатребованКолич',
       'Заказанное к-во', 'Единица измерения', 'RU Наименование',
       'TR Наименование', 'Инвентарный номер', 'Краткий текст',
       'Закуп. организация', 'Создал', 'ПоступлМатериала', 'Поступление счета',
       'Номер проекта', 'Группа материалов', 'Дата поставки',
       'Завод-поставщик', 'Вид документа', 'Статус обработки',
       'Индикатор создания', 'Стратегия деблокир.', 'НДП-материал',
       'Номер потребности', 'Объем дефицита', 'Тип даты поставки',
       'Дата деблокирования', ''Дата заявки', 'zp_zppoz', 'B3-POZ'],
      dtype='object')

In [96]:
df_me5a_cleaned['silme_durumu'] = np.where(
    df_me5a_cleaned['Заказанное к-во'].isna(),  # Eğer boşsa
    'dursun',
    np.where(
        df_me5a_cleaned['Заказанное к-во'] < df_me5a_cleaned['ЗатребованКолич'],
        'dursun',
        'sil'
    )
)

In [97]:
print(df_me5a_cleaned['silme_durumu'].value_counts())

silme_durumu
sil       287266
dursun     86158
Name: count, dtype: int64


In [98]:
df_me5a_cleaned.columns

Index(['ГОД', 'Заявка', 'Позиция заявки', 'Материал', 'Индикатор удаления_B3',
       'Затребовал', 'Индик. выдачи', 'Группа деблокир.',
       'Статус обработки заявки', 'Дата заявки', 'Группа закупок',
       'Дата изменения', 'Заказ на поставку', 'Позиция ЗкзНаПостав',
       'Дата заказа', 'Долгосрочный договор', 'ЗатребованКолич',
       'Заказанное к-во', 'Единица измерения', 'RU Наименование',
       'TR Наименование', 'Инвентарный номер', 'Краткий текст',
       'Закуп. организация', 'Создал', 'ПоступлМатериала', 'Поступление счета',
       'Номер проекта', 'Группа материалов', 'Дата поставки',
       'Завод-поставщик', 'Вид документа', 'Статус обработки',
       'Индикатор создания', 'Стратегия деблокир.', 'НДП-материал',
       'Номер потребности', 'Объем дефицита', 'Тип даты поставки',
       'Дата деблокирования', ''Дата заявки', 'zp_zppoz', 'B3-POZ',
       'silme_durumu'],
      dtype='object')

In [99]:
df_me5a_cleaned = df_me5a_cleaned.drop(
    df_me5a_cleaned[
        df_me5a_cleaned['Заказ на поставку'].notna() &
        (df_me5a_cleaned['silme_durumu'] == 'sil')
    ].index
)

In [100]:
print(df_me5a_cleaned['Заказ на поставку'].value_counts())

Заказ на поставку
3100010337    260
3100011876     66
3100013430     50
3100011264     49
3100008894     36
             ... 
3100011930      1
3100008245      1
3100015941      1
3100007553      1
3100012701      1
Name: count, Length: 572, dtype: int64


In [101]:
for col in ["Заказ на поставку", "Позиция ЗкзНаПостав"]:
    # Boş değerleri koru, dolu olanları integer yap
    df_me5a_cleaned[col] = df_me5a_cleaned[col].apply(
        lambda x: pd.NA if pd.isna(x) else int(x) if float(x).is_integer() else float(x)
    ).astype("Int64")  # nullable integer dtype

In [102]:
df_me5a_cleaned[['Заявка','Позиция заявки','Заказ на поставку','Заказанное к-во','ЗатребованКолич']].sample(n=5)

,Заявка,Позиция заявки,Заказ на поставку,Заказанное к-во,ЗатребованКолич
320528,2200004468,210,<NA>,6.0,1.0
130267,9900002494,1320,<NA>,1.0,1.0
105160,2200012576,70,<NA>,0.0,7.0
52523,9900003876,5930,<NA>,1.0,1.0
297500,2200005755,480,<NA>,0.0,2.0


In [103]:
# 1. Zorunlu temizleme işlemleri (orijinal sütunların üzerine yazar)
df_me5a_cleaned["Заявка"] = df_me5a_cleaned["Заявка"].apply(temizle_ve_cevir)
df_me5a_cleaned["Позиция заявки"] = df_me5a_cleaned["Позиция заявки"].apply(temizle_ve_cevir)

df_zmm067_2022_2025["Заявка"] = df_zmm067_2022_2025["Заявка"].apply(temizle_ve_cevir)
df_zmm067_2022_2025["Позиция заявки"] = df_zmm067_2022_2025["Позиция заявки"].apply(temizle_ve_cevir)

# 2. Anahtar oluştur
df_me5a_cleaned["zayavka_poz"] = df_me5a_cleaned["Заявка"] + "_" + df_me5a_cleaned["Позиция заявки"]
df_zmm067_2022_2025["zayavka_poz"] = df_zmm067_2022_2025["Заявка"] + "_" + df_zmm067_2022_2025["Позиция заявки"]

# 3. Dict oluştur
dict_dop = dict(zip(df_zmm067_2022_2025["zayavka_poz"], df_zmm067_2022_2025["Дополнительное"]))

# 4. map ile yeni sütun ekle
df_me5a_cleaned["Дополнительное"] = df_me5a_cleaned["zayavka_poz"].map(dict_dop)

#yeni oluşturulan sutunlara ihtiyaç kalmadı

df_me5a_cleaned = df_me5a_cleaned.drop(columns=["zayavka_poz"])
df_zmm067_2022_2025 = df_zmm067_2022_2025.drop(columns=["zayavka_poz"])

In [104]:
df_me2n_2022_2025.sample(n=5)

,ГОД,Документ закупки,Позиция,Создал,Группа закупок,Закуп. организация,Индикатор удаления,Дата документа,Наш знак,Ваш код,...,Вид докум. закупки,Код налога,Краткий текст,Дата поставки,ЕИ заказа на постав.,ТипДокумЗакупки,Тип неполноты данных,Группа материалов,Материал_ME2N,Материал_ME5A
302446,2023,3100005477,410,HALLAKOV,099,2992,<NA>,2023-12-05,AKEM155,AKEM155,...,ZSUP,V3,10UJA.KBA.TM.TB0101.22/105.M61/1,2024-01-19,КГ,F,Документ полностью сохранен,MZ-2065,4500264932,4500264932
338118,2023,3100003336,170,TEGOROVA,003,2993,<NA>,2023-08-03,HE 20019,HE 20019,...,ZSUP,VB,KLİPSLİ DÜBEL HSA-R M6X50 5/-/- 2036314,2023-08-05,ШТ,F,Документ полностью сохранен,MZ-6008,4500090395,4500090395
199228,2024,3100009180,3770,IKOVALEV,001,2991,<NA>,2024-08-21,90VFA2024,90VFA2024,...,ZSUP,VB,AKU.0130.10UMA.0.KM.TB0019.GRL1/32,2024-09-26,ШТ,F,Документ полностью сохранен,MZ-2065,4500421526,4500421526
360043,2023,3100001481,18680,RKAYNAR,109,2990,<NA>,2023-03-30,TPL-2022,TPL-2022,...,ZSUP,V3,AKU.0120.00USY.0.KM.LC0006.LG26,2023-04-28,КГ,F,Документ полностью сохранен,MZ-2065,4500196401,4500196401
249119,2024,3100007439,180,EUSLU,112,2991,<NA>,2024-04-22,HCN230010,HCN230010,...,ZSUP,VB,MATKAP UCU TE-CX 8/22 409177 Hilti,2024-04-22,ШТ,F,Документ полностью сохранен,MZ-6010,4500127939,<NA>


In [105]:
"""
del df_old_me2n
del old_employee
del df_old_me2n_filtered
del df_filtered2
"""

'\ndel df_old_me2n\ndel old_employee\ndel df_old_me2n_filtered\ndel df_filtered2\n'

In [106]:
df_old_me2n=df_me2n_2022_2025.copy()

In [107]:
df_old_me2n.columns

Index(['ГОД', 'Документ закупки', 'Позиция', 'Создал', 'Группа закупок',
       'Закуп. организация', 'Индикатор удаления', 'Дата документа',
       'Наш знак', 'Ваш код', 'Имя поставщика', 'Заявка', 'Позиция заявки',
       'Материал', 'Инвентарный номер', 'Наименование RU', 'Наименование TR',
       'Объем заказа', 'Складская ЕИ', 'еще поставить (количество)',
       'Еще для поставки (стоимость)', 'Уже поставлено (количество)',
       'Для фактурирования (стоим.)', 'Для фактурирования (колич.)',
       'Количество в СЕИ', 'Единица цены', 'СтоимЗаказа нетто', 'Валюта',
       'Цена нетто', 'Вид докум. закупки', 'Код налога', 'Краткий текст',
       'Дата поставки', 'ЕИ заказа на постав.', 'ТипДокумЗакупки',
       'Тип неполноты данных', 'Группа материалов', 'Материал_ME2N',
       'Материал_ME5A'],
      dtype='object')

In [108]:
df_old_me2n['Создал'].unique()

array(['MAKIF', 'ESAVRAN', 'NBEREZOVSKAY', 'MCIMAN', 'NOZKAN', 'HALLAKOV',
       'ACHARYEV', 'APOTAPOV', 'ISVIRID', 'ZIZZET', 'FPINAR', 'RKUCUK',
       'EKOC', 'MSAHIN', 'KGAIVUK', 'TEGOROVA', 'VKUZMICHEV',
       'ILUKIANCHUK', 'SISIGUZEL', 'MMASLOV', 'IBATENKO', 'DKILIC',
       'SSTRIAPUKHIN', 'TESTWH', 'OSKREBETS', 'ETROFIMUSHK', 'FSALIMOV',
       'RDAVLETBAKOV', 'NARIKAN', 'ZDURMAZ', 'ESAMSONOVA', <NA>,
       'SUBKHANGULOV', 'SOZEVIN', 'IKALASHNIKOV', 'ZHUSEYNOVA',
       'RGAVRIKOVA', 'ARTGALIEV', 'ILAMBIN', 'NOMAROVA', 'SZHELAMSKII',
       'NBULUT', 'ESIKORSKAIA', 'IKOVALEV', 'MCHEREMNYKH', 'NZYBINSSKAYA',
       'SUSLU', 'MPETROVA', 'TZADOROZHN', 'SBASTANOV', 'MZHAGIPAROV',
       'IABBASOVA', 'MCETIN', 'KLITVINOVA', 'SYILDIRIM', 'GCEREZCI',
       'NKANYSBAY', 'ACOSKUN', 'OLUGOVIK', 'YOZTURK', 'FANNAKULIEVA',
       'IMURADIAN', 'TMISHATKINA', 'GTORAMAN', 'MOZCAN', 'RKAYNAR',
       'EUSLU', 'VSOYBIR', 'OKOKOGLU', 'VPIROG', 'MLEBEDEV',
       'ZABDUVAKHAP', 'ESARAL', 'SMA

In [109]:
"""
old_employee =['TURMANOV','ILAMBIN','IKALASHNIKOV','MOZCAN','ZHUSEYNOVA','ESIKORSKAIA','FANNAKULIEVA',
               'EUSLU','RALIOGLU','GTORAMAN','YOZTURK','GCEREZCI','MMAMAZHANOV','ESARAL','KDUVERLIOGLU',
               'AMARIA','MPETROVA','NKANYSBAY','ACOSKUN','RKAYNAR','EMILAEVA',
               'DPANTEEV','EYILDIZ','VSOYBIR','DSHARKOV','CCANBULAT','VPIROG',
               'TDZHURAEV','TMISHATKINA','MMOROZOV','MMURADOV'
              ]
              """

"\nold_employee =['TURMANOV','ILAMBIN','IKALASHNIKOV','MOZCAN','ZHUSEYNOVA','ESIKORSKAIA','FANNAKULIEVA',\n               'EUSLU','RALIOGLU','GTORAMAN','YOZTURK','GCEREZCI','MMAMAZHANOV','ESARAL','KDUVERLIOGLU',\n               'AMARIA','MPETROVA','NKANYSBAY','ACOSKUN','RKAYNAR','EMILAEVA',\n               'DPANTEEV','EYILDIZ','VSOYBIR','DSHARKOV','CCANBULAT','VPIROG',\n               'TDZHURAEV','TMISHATKINA','MMOROZOV','MMURADOV'\n              ]\n              "

In [110]:
old_employee =['NBEREZOVSKAY', 'NOZKAN', 'DKILIC', 'ESAVRAN', 'HALLAKOV',
       'ACHARYEV', 'MCIMAN', 'APOTAPOV', 'TEGOROVA', 'EKOC', 'IBATENKO',
       'ZIZZET', 'SISIGUZEL', 'FPINAR', 'KGAIVUK', 'MAKIF', 'MSAHIN',
       'FSALIMOV', 'SSTRIAPUKHIN', 'VKUZMICHEV', 'ETROFIMUSHK',
       'ILUKIANCHUK', 'ZDURMAZ', 'ISVIRID', 'ESAMSONOVA', 'SOZEVIN',
       'NARIKAN', 'IKALASHNIKOV', 'ZHUSEYNOVA', 'RGAVRIKOVA', 'ARTGALIEV',
       'ILAMBIN', 'NOMAROVA', 'SZHELAMSKII', 'NBULUT', 'ESIKORSKAIA',
       'IKOVALEV', 'MCHEREMNYKH', 'NZYBINSSKAYA', 'SUSLU', 'OSKREBETS',
       'TZADOROZHN', 'MPETROVA', 'SBASTANOV', 'MZHAGIPAROV', 'IABBASOVA',
       'MCETIN', 'KLITVINOVA', 'SYILDIRIM', 'GCEREZCI', 'NKANYSBAY',
       'ACOSKUN', 'RDAVLETBAKOV', 'OLUGOVIK', 'YOZTURK', 'FANNAKULIEVA',
       'TMISHATKINA', 'IMURADIAN', 'GTORAMAN', 'MOZCAN', 'RKAYNAR',
       'EUSLU', 'VSOYBIR', 'OKOKOGLU', 'VPIROG', 'MLEBEDEV',
       'ZABDUVAKHAP', 'ESARAL', 'SMANAFOVA', 'TURMANOV', 'EMILAEVA',
       'AMARIA', 'AISAKINA', 'ACHERKASOV', 'CCANBULAT', 'EYILDIZ',
       'MANDREEV', 'EPULATOV', 'MMOROZOV', 'MMAMAZHANOV', 'KDUVERLIOGLU',
       'TDZHURAEV', 'NTUAEVA', 'TAPARINA', 'IKUCUKKAVRUK', 'DSHARKOV',
       'AEIGOSHEV', 'IZEMLIAKOVA', 'MMURADOV', 'MBURMAKOVA', 'SBAKHITOV',
       'EGASIMOV', 'SURMANTSEVA', 'AEROKHOV', 'NHASANLI', 'MVOLKOVA',
       'RMAMMADOV', 'LUFITSEVA', 'IISMAILOV', 'EKHUDIAKOVA',
       'FMUSTAFAZADE', 'RALIOGLU', 'DPANTEEV', 'ZBONDAREVA'
              ]

In [111]:
df_old_me2n = df_old_me2n[df_old_me2n['Создал'].isin(old_employee)]

In [112]:
df_old_me2n.columns

Index(['ГОД', 'Документ закупки', 'Позиция', 'Создал', 'Группа закупок',
       'Закуп. организация', 'Индикатор удаления', 'Дата документа',
       'Наш знак', 'Ваш код', 'Имя поставщика', 'Заявка', 'Позиция заявки',
       'Материал', 'Инвентарный номер', 'Наименование RU', 'Наименование TR',
       'Объем заказа', 'Складская ЕИ', 'еще поставить (количество)',
       'Еще для поставки (стоимость)', 'Уже поставлено (количество)',
       'Для фактурирования (стоим.)', 'Для фактурирования (колич.)',
       'Количество в СЕИ', 'Единица цены', 'СтоимЗаказа нетто', 'Валюта',
       'Цена нетто', 'Вид докум. закупки', 'Код налога', 'Краткий текст',
       'Дата поставки', 'ЕИ заказа на постав.', 'ТипДокумЗакупки',
       'Тип неполноты данных', 'Группа материалов', 'Материал_ME2N',
       'Материал_ME5A'],
      dtype='object')

In [113]:
df_old_me2n['Индикатор удаления'].nunique()

2

In [114]:
df_old_me2n=df_old_me2n[(df_old_me2n['Еще для поставки (стоимость)']>0) & (df_old_me2n['Индикатор удаления'].isna())]

In [115]:
df_old_me2n['Индикатор удаления'].nunique()

0

In [116]:
# Gruplama için kullanılacak sütunlar
group_cols = ['Документ закупки']
# Aggregation (toplama/birleştirme) fonksiyonları
agg_funcs = {

    'Создал':'first',
    'Группа закупок':'first' ,
    'Закуп. организация':'first',
    'Дата документа':'first',
    'Наш знак':'first'   ,
    'Ваш код':'first',
    'Имя поставщика':'first',
    'СтоимЗаказа нетто':'sum',
    'Еще для поставки (стоимость)':'sum',
    'Валюта':'first'
}

# Gruplama ve aggregation işlemini gerçekleştiriyoruz
df_old_me2n_filtered = df_old_me2n.groupby(group_cols, as_index=False).agg(agg_funcs)

# The filtering for 'Индикатор удаления' was already performed on df_old_me2n
df_old_me2n_filtered=df_old_me2n_filtered[df_old_me2n_filtered['Еще для поставки (стоимость)']>0]

In [117]:
df_old_me2n_filtered = df_old_me2n_filtered.drop(columns=['Имя поставщика'])

In [118]:
df_old_me2n_filtered.columns

Index(['Документ закупки', 'Создал', 'Группа закупок', 'Закуп. организация',
       'Дата документа', 'Наш знак', 'Ваш код', 'СтоимЗаказа нетто',
       'Еще для поставки (стоимость)', 'Валюта'],
      dtype='object')

In [119]:
df_calisilan['Поставщик']

,Поставщик
0,<NA>
1,<NA>
2,<NA>
3,<NA>
4,<NA>
...,...
816606,1000019187
816607,1000019187
816608,1000019187
816609,1000001070


In [120]:
df_calisilan.columns

Index(['ГОД', 'zp_zppoz', 'B3-B3POZ', 'Документ_закупки', 'Зп_Поз',
       'Индикатор удаления', 'неполн.', 'Материал', 'Заявка', 'Позиция заявки',
       'RU Наименование', 'TR Наименование', 'Инвентарный номер в SAS',
       'Количество ЗП', 'Минимальная цена ЗП', 'Минимальная цена ЗП.1',
       'Минимальная цена ЗП.2', 'Единица ЗП', 'Цена нетто ЗП',
       'Итого цена нетто', 'Валюта ЗП', 'Вид докум. закупки', 'Дата создания',
       'Дата поставки', 'Дата доставки, одобренная Поставщиком',
       'Группа закупок', 'Название ГрЗакупок', 'БЕ', 'Название фирмы',
       'Закуп. организация', 'Название ЗакупОрг', 'Завод', 'Имя 1',
       'Поставщик', 'Группа материалов', 'Название группы', 'Краткий текст',
       'Создал', 'Вид докум. закупки.1', 'Обозначение ВидДокум',
       'Дата создания.1', 'Создал.1', 'Группа закупок.1',
       'Название ГрЗакупок.1', 'ЕИ заказа на постав.', 'Единица цены',
       'Цена заказа нетто', 'Стоимость брутто', 'Валюта',
       'Недопоставленное количест

In [121]:
# 1. Anahtar sütunları tuple haline getirelim
df_calisilan["key"] = df_calisilan[["Документ_закупки"]].apply(tuple, axis=1)
df_old_me2n_filtered["key"] = df_old_me2n_filtered[["Документ закупки"]].apply(tuple, axis=1)

# 2. Aktarılacak sütunlar ve hedef adları
map_dict_B3_status = {
    "имя_поставщика":"имя_поставщика",
    "Поставщик":"Номер Поставщика SAP"
}

# 3. Her sütun için dict map uygulama
for source_col, target_col in map_dict_B3_status.items():
    temp_dict = dict(zip(df_calisilan["key"], df_calisilan[source_col]))
    df_old_me2n_filtered[target_col] = df_old_me2n_filtered["key"].map(temp_dict)

# 4. Geçici key sütunlarını kaldır
df_old_me2n_filtered.drop(columns=["key"], inplace=True)
df_calisilan.drop(columns=["key"], inplace=True)

In [122]:
df_old_me2n_filtered.sample(n=5)

,Документ закупки,Создал,Группа закупок,Закуп. организация,Дата документа,Наш знак,Ваш код,СтоимЗаказа нетто,Еще для поставки (стоимость),Валюта,имя_поставщика,Номер Поставщика SAP
1391,3100009003,TEGOROVA,003,2993,2024-08-19,Спец 217,AK-NPEM-2920,3198952.98,3198952.98,USD,AERKLIMA MUHENDISLIK TICARET VE SAN,1000001414
73,3100001296,RKAYNAR,109,2991,2023-03-20,AK-C-81,AK-C-81,3328.71,3328.71,USD,ÇELKONSAN MAK. SAN. TİC. A.Ş.,1000017037
2137,3100012290,APOTAPOV,003,2993,2025-03-05,SPEKNo428,TSM07221507,181652.65,181652.65,USD,AERKLIMA MUHENDISLIK TICARET VE SAN,1000001414
1411,3100009153,IKOVALEV,001,2991,2024-08-21,90VFA2024,90VFA2024,24473.69,24473.69,USD,IC IÇTAS ENDÜSTRIYEL BORU LOJISTIK,2020
1151,3100007753,MZHAGIPAROV,002,2992,2024-05-20,AK-EM-137,AK-EM-137,1755.19,1755.19,USD,INCE IS METAL PROJE INSAAT MAKINA,1000001420


In [123]:
#her bir kuratörü tek tek kaydetme
# 1. Gerekli sütun sırasını tanımla
columns_order = [
    'Создал','Группа закупок','Документ закупки','Закуп. организация','Дата документа','Наш знак','Ваш код','имя_поставщика','Номер Поставщика SAP','СтоимЗаказа нетто','Еще для поставки (стоимость)','Валюта'
]

# 2. Benzersiz Заявитель değerlerini al
employees = df_old_me2n_filtered['Создал'].dropna().unique()

# 3. Çıktıları kaydetmek için klasör oluştur (varsa geç)
os.makedirs("not_delivered", exist_ok=True)

# 4. Her Заявитель için döngü
for z_name in employees:
    # Dosya adı için geçersiz karakterleri temizle
    safe_name = "".join(c if c.isalnum() or c in "._-" else "_" for c in z_name)

    # Filtreleme ve sütun sıralama
    df_filtered2 = df_old_me2n_filtered[df_old_me2n_filtered['Создал'] == z_name].copy()

    # Sütun kontrolü: eksik olanları atla (varsa uyarı verir)
    missing_cols = [col for col in columns_order if col not in df_filtered2.columns]
    if missing_cols:
        print(f"{z_name} için eksik sütun(lar): {missing_cols}")
        continue

    df_filtered2 = df_filtered2[columns_order]

    # Excel olarak kaydet
    output_path = f"not_delivered/me2n_notdelivered_{safe_name}.xlsx"
    df_filtered2.to_excel(output_path, index=False)

print("✅ Tüm dosyalar 'not_delivered/' klasörüne kaydedildi.")

✅ Tüm dosyalar 'not_delivered/' klasörüne kaydedildi.


In [124]:
zip_filename = "not_delivered_me2n_all.zip"

with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk("not_delivered"):
        for file in files:
            if file.endswith(".xlsx"):
                file_path = os.path.join(root, file)
                zipf.write(file_path)

In [125]:
from google.colab import files
files.download(zip_filename)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [126]:
#sadece geliştirme aşamasında kullanılacak !
"""
del zmb51_dummy
del df_B3_status
del zmm059_dummy
del me2n_dummy
del zpp001_dummy

"""

'\ndel zmb51_dummy\ndel df_B3_status\ndel zmm059_dummy\ndel me2n_dummy\ndel zpp001_dummy\n\n'

In [127]:
df_B3_status=df_me5a_2022_2025.copy()
zmb51_dummy=birlesik_df_zmb51.copy()
zmm059_dummy=df_calisilan.copy()
me2n_dummy=df_me2n_2022_2025.copy()
zpp001_dummy=zpp001.copy()

In [128]:
zmb51_dummy.columns

Index(['ГОД', 'Вид движения', 'Партия', 'Материал', 'Группа материалов',
       'Заказ на поставку', 'Позиция', 'Наш знак', 'Ваш код',
       'Внутренний заказ', 'Позиция заявки', 'Склад', 'Дата ввода',
       'Время ввода', 'Дата документа', 'Дата проводки', 'Документ материала',
       'Поз. док. материала', 'Признак 2', 'Признак 5', 'Ссылка',
       'Краткий текст материала', 'Кол-во в ЕИ ввода', 'Объем заказа',
       'ЕИ ввода', 'Сумма во ВВ', 'Количество', 'Поставщик', 'Имя поставщика',
       'Накладная', 'Текст заголовка документа', 'Заявитель',
       'Инвентарный номер', 'Имя пользователя', 'БЕ', 'Базовая ЕИ',
       'ГодДокумМатериала', 'Валюта', 'Завод', 'Имя 1', 'Основное средство',
       'ЕИ цены заказа', 'ЕИ заказа на постав.'],
      dtype='object')

In [129]:
display(zmb51_dummy[(zmb51_dummy["Заказ на поставку"]==3100005467) & (zmb51_dummy["Позиция"]==80)][["Кол-во в ЕИ ввода", "Сумма во ВВ", "Наш знак"]])

,Кол-во в ЕИ ввода,Сумма во ВВ,Наш знак
308513,1.0,14511.18,AK-C-389


In [130]:
#zmb51_dummy = duzenle_sayi_sutunu(zmb51_dummy, "Кол-во в ЕИ ввода")

In [131]:
zmb51_dummy['Признак 2&кол-во&ЕИ']=zmb51_dummy['Признак 2'].astype(str)+' '+zmb51_dummy['Кол-во в ЕИ ввода'].astype(str)+' '+zmb51_dummy['ЕИ ввода'].astype(str)
zmb51_dummy['Ссылка&кол-во&ЕИ']=zmb51_dummy['Ссылка'].astype(str)+' '+zmb51_dummy['Кол-во в ЕИ ввода'].astype(str)+' '+zmb51_dummy['ЕИ ввода'].astype(str)
zmb51_dummy["Кол-во в ЕИ ввода"] = pd.to_numeric(zmb51_dummy["Кол-во в ЕИ ввода"], errors="coerce").astype("float64")


In [132]:
zpp001_dummy.columns

Index(['﻿Валюта оплаты', 'Номер заявки на платеж', 'Тип транзакции',
       'Тип документа', 'Тип платежа', 'Статья ДДС', 'Наименование',
       'Purpose Code', 'Протокол', 'Номер договора', 'Инвойс', 'Кредитор',
       'Название кредитора', 'Оплачиваемая сумма', 'Валюта', 'Автор изменения',
       'Дата изменения', 'Создал', 'Дата создания', 'Unnamed: 19',
       'Статус согласующего', 'Оплаченная Сумма', 'Unnamed: 22', 'Дебитор',
       'Название заказчика', 'Unnamed: 25', 'Документ закупки', 'Unnamed: 27',
       'Unnamed: 28', 'Статус', 'Время', 'Время изменения', 'Дата платежа',
       'fat_ted', 'Номер_счета', 'Дата_документа_счета', 'зп_поз'],
      dtype='object')

In [133]:
df_calisilan.columns

Index(['ГОД', 'zp_zppoz', 'B3-B3POZ', 'Документ_закупки', 'Зп_Поз',
       'Индикатор удаления', 'неполн.', 'Материал', 'Заявка', 'Позиция заявки',
       'RU Наименование', 'TR Наименование', 'Инвентарный номер в SAS',
       'Количество ЗП', 'Минимальная цена ЗП', 'Минимальная цена ЗП.1',
       'Минимальная цена ЗП.2', 'Единица ЗП', 'Цена нетто ЗП',
       'Итого цена нетто', 'Валюта ЗП', 'Вид докум. закупки', 'Дата создания',
       'Дата поставки', 'Дата доставки, одобренная Поставщиком',
       'Группа закупок', 'Название ГрЗакупок', 'БЕ', 'Название фирмы',
       'Закуп. организация', 'Название ЗакупОрг', 'Завод', 'Имя 1',
       'Поставщик', 'Группа материалов', 'Название группы', 'Краткий текст',
       'Создал', 'Вид докум. закупки.1', 'Обозначение ВидДокум',
       'Дата создания.1', 'Создал.1', 'Группа закупок.1',
       'Название ГрЗакупок.1', 'ЕИ заказа на постав.', 'Единица цены',
       'Цена заказа нетто', 'Стоимость брутто', 'Валюта',
       'Недопоставленное количест

In [134]:
zpp001_dummy["Документ закупки"] = zpp001_dummy["Документ закупки"].apply(temizle_ve_cevir)
zpp001_dummy["зп_поз"] = zpp001_dummy["зп_поз"].apply(temizle_ve_cevir)
zpp001_dummy = zpp001_dummy.applymap(temizle_nan)
df_calisilan["Документ_закупки"] = df_calisilan["Документ_закупки"].apply(temizle_ve_cevir)
df_calisilan["Зп_Поз"] = df_calisilan["Зп_Поз"].apply(temizle_ve_cevir)
df_calisilan = df_calisilan.applymap(temizle_nan)

/tmp/ipykernel_44620/3926105340.py:3: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  zpp001_dummy = zpp001_dummy.applymap(temizle_nan)
/tmp/ipykernel_44620/3926105340.py:6: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_calisilan = df_calisilan.applymap(temizle_nan)


In [135]:
# 1. Anahtar sütunları tuple haline getirelim
zpp001_dummy["key"] = zpp001_dummy[["Документ закупки", "зп_поз"]].apply(tuple, axis=1)
df_calisilan["key"] = df_calisilan[["Документ_закупки", "Зп_Поз"]].apply(tuple, axis=1)

# 2. Aktarılacak sütunlar ve hedef adları
map_dict_B3_status = {
    "Заявка": "ВЗ",
    "Позиция заявки":"ВЗ_ПОЗ",


}

# 3. Her sütun için dict map uygulama
for source_col, target_col in map_dict_B3_status.items():
    temp_dict = dict(zip(df_calisilan["key"], df_calisilan[source_col]))
    zpp001_dummy[target_col] = zpp001_dummy["key"].map(temp_dict)

# 4. Geçici key sütunlarını kaldır
df_calisilan.drop(columns=["key"], inplace=True)
zpp001_dummy.drop(columns=["key"], inplace=True)

In [136]:
zpp001_dummy.columns

Index(['﻿Валюта оплаты', 'Номер заявки на платеж', 'Тип транзакции',
       'Тип документа', 'Тип платежа', 'Статья ДДС', 'Наименование',
       'Purpose Code', 'Протокол', 'Номер договора', 'Инвойс', 'Кредитор',
       'Название кредитора', 'Оплачиваемая сумма', 'Валюта', 'Автор изменения',
       'Дата изменения', 'Создал', 'Дата создания', 'Unnamed: 19',
       'Статус согласующего', 'Оплаченная Сумма', 'Unnamed: 22', 'Дебитор',
       'Название заказчика', 'Unnamed: 25', 'Документ закупки', 'Unnamed: 27',
       'Unnamed: 28', 'Статус', 'Время', 'Время изменения', 'Дата платежа',
       'fat_ted', 'Номер_счета', 'Дата_документа_счета', 'зп_поз', 'ВЗ',
       'ВЗ_ПОЗ'],
      dtype='object')

In [137]:
zpp001_dummy[["Документ закупки","зп_поз","ВЗ","ВЗ_ПОЗ"]].sample(n=5)

,Документ закупки,зп_поз,ВЗ,ВЗ_ПОЗ
5545,<NA>,<NA>,2200000450.0,680.0
5280,<NA>,<NA>,2200000450.0,680.0
3009,<NA>,<NA>,2200000450.0,680.0
441,3100007222,1470,<NA>,<NA>
1017,<NA>,<NA>,2200000450.0,680.0


In [138]:
#<zx<zxz<x

In [139]:
zmb51_dummy.columns

Index(['ГОД', 'Вид движения', 'Партия', 'Материал', 'Группа материалов',
       'Заказ на поставку', 'Позиция', 'Наш знак', 'Ваш код',
       'Внутренний заказ', 'Позиция заявки', 'Склад', 'Дата ввода',
       'Время ввода', 'Дата документа', 'Дата проводки', 'Документ материала',
       'Поз. док. материала', 'Признак 2', 'Признак 5', 'Ссылка',
       'Краткий текст материала', 'Кол-во в ЕИ ввода', 'Объем заказа',
       'ЕИ ввода', 'Сумма во ВВ', 'Количество', 'Поставщик', 'Имя поставщика',
       'Накладная', 'Текст заголовка документа', 'Заявитель',
       'Инвентарный номер', 'Имя пользователя', 'БЕ', 'Базовая ЕИ',
       'ГодДокумМатериала', 'Валюта', 'Завод', 'Имя 1', 'Основное средство',
       'ЕИ цены заказа', 'ЕИ заказа на постав.', 'Признак 2&кол-во&ЕИ',
       'Ссылка&кол-во&ЕИ'],
      dtype='object')

In [140]:
# Gruplama için kullanılacak sütunlar
group_cols = ['Заказ на поставку', 'Позиция']
# Aggregation (toplama/birleştirme) fonksiyonları
agg_funcs = {
    'Кол-во в ЕИ ввода': 'sum', # 'Кол-во в ЕИ ввода' sütunundaki değerleri topluyoruz
    'Дата проводки': lambda x: ';'.join(x.dropna().astype(str)),
    # 'Дата проводки' sütununu 'YYYY-MM-DD' formatında stringe çevirip,
    # benzersiz değerleri alıp ';' ile birleştiriyoruz. NaN değerler dışarıda bırakılır.
    'Признак 2&кол-во&ЕИ': lambda x: ';'.join(x.dropna().astype(str)),
    # 'накладная&кол-во&ЕИ 1' sütunundaki benzersiz string değerleri ';' ile birleştiriyoruz. NaN değerler dışarıda bırakılır.
    'Ссылка&кол-во&ЕИ': lambda x: ';'.join(x.dropna().astype(str))
    # 'накладная&кол-во&ЕИ 2' sütunundaki benzersiz string değerleri ';' ile birleştiriyoruz. NaN değerler dışarıda bırakılır.
}

# Gruplama ve aggregation işlemini gerçekleştiriyoruz
result_df = zmb51_dummy.groupby(group_cols, as_index=False).agg(agg_funcs)
del zmb51_dummy
zmb51_dummy=result_df.copy()
del result_df

In [141]:
zmb51_dummy["Заказ на поставку"] = zmb51_dummy["Заказ на поставку"].apply(temizle_ve_cevir)
zmb51_dummy["Позиция"] = zmb51_dummy["Позиция"].apply(temizle_ve_cevir)
me2n_dummy["Документ закупки"] = me2n_dummy["Документ закупки"].apply(temizle_ve_cevir)
me2n_dummy["Позиция"] = me2n_dummy["Позиция"].apply(temizle_ve_cevir)

In [142]:
me2n_dummy.columns

Index(['ГОД', 'Документ закупки', 'Позиция', 'Создал', 'Группа закупок',
       'Закуп. организация', 'Индикатор удаления', 'Дата документа',
       'Наш знак', 'Ваш код', 'Имя поставщика', 'Заявка', 'Позиция заявки',
       'Материал', 'Инвентарный номер', 'Наименование RU', 'Наименование TR',
       'Объем заказа', 'Складская ЕИ', 'еще поставить (количество)',
       'Еще для поставки (стоимость)', 'Уже поставлено (количество)',
       'Для фактурирования (стоим.)', 'Для фактурирования (колич.)',
       'Количество в СЕИ', 'Единица цены', 'СтоимЗаказа нетто', 'Валюта',
       'Цена нетто', 'Вид докум. закупки', 'Код налога', 'Краткий текст',
       'Дата поставки', 'ЕИ заказа на постав.', 'ТипДокумЗакупки',
       'Тип неполноты данных', 'Группа материалов', 'Материал_ME2N',
       'Материал_ME5A'],
      dtype='object')

In [143]:
# 1. Anahtar sütunları tuple haline getirelim
me2n_dummy["key"] = me2n_dummy[["Документ закупки", "Позиция"]].apply(tuple, axis=1)
zmb51_dummy["key"] = zmb51_dummy[["Заказ на поставку", "Позиция"]].apply(tuple, axis=1)

# 2. Aktarılacak sütunlar ve hedef adları
map_dict_B3_status = {
    "Кол-во в ЕИ ввода": "Кол-во в ЕИ ввода",
    "Дата проводки":"Дата проводки",
    "Признак 2&кол-во&ЕИ":"Признак 2&кол-во&ЕИ",
    "Ссылка&кол-во&ЕИ":"Ссылка&кол-во&ЕИ",


}

# 3. Her sütun için dict map uygulama
for source_col, target_col in map_dict_B3_status.items():
    temp_dict = dict(zip(zmb51_dummy["key"], zmb51_dummy[source_col]))
    me2n_dummy[target_col] = me2n_dummy["key"].map(temp_dict)

# 4. Geçici key sütunlarını kaldır
zmb51_dummy.drop(columns=["key"], inplace=True)
me2n_dummy.drop(columns=["key"], inplace=True)

In [144]:
zmm059_dummy["Документ_закупки"] = zmm059_dummy["Документ_закупки"].apply(temizle_ve_cevir)
zmm059_dummy["Зп_Поз"] = zmm059_dummy["Зп_Поз"].apply(temizle_ve_cevir)
me2n_dummy["Документ закупки"] = me2n_dummy["Документ закупки"].apply(temizle_ve_cevir)
me2n_dummy["Позиция"] = me2n_dummy["Позиция"].apply(temizle_ve_cevir)

In [145]:
# 1. Anahtar sütunları tuple haline getirelim
me2n_dummy["key"] = me2n_dummy[["Документ закупки", "Позиция"]].apply(tuple, axis=1)
zmm059_dummy["key"] = zmm059_dummy[["Документ_закупки", "Зп_Поз"]].apply(tuple, axis=1)

# 2. Aktarılacak sütunlar ve hedef adları
map_dict_56 = {
    'Группа закупок':'Группа закупок_me2n',
    'Закуп. организация':'Закуп. организация_me2n',
    'Наш знак':'Наш знак_me2n',
    'Ваш код':'Ваш код_me2n',
    'Объем заказа':'Объем заказа_me2n',
    'Материал':'Материал_me2n',
    "Складская ЕИ":"Складская ЕИ_me2n",
    "ЕИ заказа на постав.":"ЕИ заказа на постав._me2n",
    "Кол-во в ЕИ ввода": "Кол-во в ЕИ ввода",
    "Дата проводки":"Дата проводки_me2n",
    "Признак 2&кол-во&ЕИ":"Признак 2&кол-во&ЕИ",
    "Ссылка&кол-во&ЕИ":"Ссылка&кол-во&ЕИ",



}

# 3. Her sütun için dict map uygulama
for source_col, target_col in map_dict_56.items():
    temp_dict = dict(zip(me2n_dummy["key"], me2n_dummy[source_col]))
    zmm059_dummy[target_col] = zmm059_dummy["key"].map(temp_dict)

# 4. Geçici key sütunlarını kaldır
zmm059_dummy.drop(columns=["key"], inplace=True)
me2n_dummy.drop(columns=["key"], inplace=True)

In [146]:
zmm059_dummy["Закуп. организация_me2n"]=zmm059_dummy["Закуп. организация_me2n"].apply(temizle_ve_cevir)

In [147]:
zmm059_dummy.loc[zmm059_dummy["Индикатор удаления"].notna(), "Объем заказа_me2n"] = pd.NA

In [148]:
zmm059_dummy.loc[zmm059_dummy["Индикатор удаления"].notna(), "Индикатор удаления"] = zmm059_dummy["Индикатор удаления"]+"_"+zmm059_dummy["Документ_закупки"]+"-"+zmm059_dummy["Зп_Поз"]

In [149]:
zmm059_dummy[["Индикатор удаления","Объем заказа_me2n"]].sample(n=5 , random_state=42)

,Индикатор удаления,Объем заказа_me2n
376934,<NA>,NaN
107739,<NA>,NaN
141694,<NA>,1.0
547189,<NA>,NaN
209670,<NA>,1.0


In [150]:
zmm059_dummy.columns

Index(['ГОД', 'zp_zppoz', 'B3-B3POZ', 'Документ_закупки', 'Зп_Поз',
       'Индикатор удаления', 'неполн.', 'Материал', 'Заявка', 'Позиция заявки',
       'RU Наименование', 'TR Наименование', 'Инвентарный номер в SAS',
       'Количество ЗП', 'Минимальная цена ЗП', 'Минимальная цена ЗП.1',
       'Минимальная цена ЗП.2', 'Единица ЗП', 'Цена нетто ЗП',
       'Итого цена нетто', 'Валюта ЗП', 'Вид докум. закупки', 'Дата создания',
       'Дата поставки', 'Дата доставки, одобренная Поставщиком',
       'Группа закупок', 'Название ГрЗакупок', 'БЕ', 'Название фирмы',
       'Закуп. организация', 'Название ЗакупОрг', 'Завод', 'Имя 1',
       'Поставщик', 'Группа материалов', 'Название группы', 'Краткий текст',
       'Создал', 'Вид докум. закупки.1', 'Обозначение ВидДокум',
       'Дата создания.1', 'Создал.1', 'Группа закупок.1',
       'Название ГрЗакупок.1', 'ЕИ заказа на постав.', 'Единица цены',
       'Цена заказа нетто', 'Стоимость брутто', 'Валюта',
       'Недопоставленное количест

In [151]:
# Gruplama için kullanılacak sütunlar
group_cols = ['Заявка', 'Позиция заявки']
# Aggregation (toplama/birleştirme) fonksiyonları
agg_funcs = {

    'Документ_закупки': lambda x: ';'.join(x.dropna().astype(str)),
    'Зп_Поз': lambda x : ';'.join(x.dropna().astype(str)),
    'Индикатор удаления': lambda x : ';'.join(x.dropna().astype(str)),
    'имя_поставщика': lambda x : ';'.join(x.dropna().astype(str)),
    'Группа закупок_me2n': lambda x : ';'.join(x.dropna().astype(str)),
    'Закуп. организация_me2n': lambda x : ';'.join(x.dropna().astype(str)),
    'Наш знак_me2n': lambda x : ';'.join(x.dropna().astype(str)),
    'Ваш код_me2n': lambda x : ';'.join(x.dropna().astype(str)),
    'Объем заказа_me2n': 'sum',
    'Материал_me2n': lambda x : ';'.join(x.dropna().astype(str)),
    'Складская ЕИ_me2n': lambda x : ';'.join(x.dropna().astype(str)),
    'ЕИ заказа на постав._me2n': lambda x : ';'.join(x.dropna().astype(str)),
    'Кол-во в ЕИ ввода': 'sum',
    'Признак 2&кол-во&ЕИ': lambda x : ';'.join(x.dropna().astype(str)),
    'Ссылка&кол-во&ЕИ': lambda x : ';'.join(x.dropna().astype(str)),
    'Дата проводки_me2n': lambda x : ';'.join(x.dropna().astype(str))
}

# Gruplama ve aggregation işlemini gerçekleştiriyoruz
result_df = zmm059_dummy.groupby(group_cols, as_index=False).agg(agg_funcs)
del zmm059_dummy
zmm059_dummy=result_df.copy()
del result_df

In [152]:
zmm059_dummy["Заявка"] = zmm059_dummy["Заявка"].apply(temizle_ve_cevir)
zmm059_dummy["Позиция заявки"] = zmm059_dummy["Позиция заявки"].apply(temizle_ve_cevir)
df_B3_status["Заявка"] = df_B3_status["Заявка"].apply(temizle_ve_cevir)
df_B3_status["Позиция заявки"] = df_B3_status["Позиция заявки"].apply(temizle_ve_cevir)

In [153]:
df_B3_status=df_B3_status.drop(columns=['Объем заказа','Имя поставщика',
                                        'Наш знак', 'Цена нетто', 'Валюта', 'ЕИ заказа на постав.',
                                        'Единица цены', 'Заказ на поставку', 'Позиция ЗкзНаПостав',
                                        'Индикатор удаления_зп','Индик. выдачи','Статус обработки заявки',
                                        'Дата изменения', 'Дата заказа','Долгосрочный договор','Закуп. организация','Создал',
                                        'ПоступлМатериала', 'Поступление счета', 'Номер проекта',
                                        'Дата поставки', 'Завод-поставщик','Вид документа', 'Статус обработки', 'Индикатор создания',
                                        'Стратегия деблокир.', 'НДП-материал', 'Номер потребности',
                                        'Объем дефицита', 'Тип даты поставки', 'Дата деблокирования','B3-B3POZ'

                                        ])

In [154]:
# 1. Anahtar sütunları tuple haline getirelim
df_B3_status["key"] = df_B3_status[["Заявка", "Позиция заявки"]].apply(tuple, axis=1)
zmm059_dummy["key"] = zmm059_dummy[["Заявка", "Позиция заявки"]].apply(tuple, axis=1)

# 2. Aktarılacak sütunlar ve hedef adları
map_dict_B3_status = {
    "Документ_закупки":"Документ_закупки",
    "Зп_Поз":"Зп_Поз",
    "Индикатор удаления":"Индикатор удаления",
    "имя_поставщика":"имя_поставщика",
    "Группа закупок_me2n":"Группа закупок_me2n",
    "Закуп. организация_me2n":"Закуп. организация_me2n",
    "Наш знак_me2n":"Наш знак_me2n",
    "Ваш код_me2n":"Ваш код_me2n",
    "Объем заказа_me2n":"Объем заказа_me2n",
    "Материал_me2n":"Материал_me2n",
    "Складская ЕИ_me2n":"Складская ЕИ_me2n",
    "ЕИ заказа на постав._me2n":"ЕИ заказа на постав._me2n",
    "Кол-во в ЕИ ввода":"Кол-во в ЕИ ввода",
    "Дата проводки_me2n":"Дата проводки",
    "Признак 2&кол-во&ЕИ":"Признак 2&кол-во&ЕИ",
    "Ссылка&кол-во&ЕИ":"Ссылка&кол-во&ЕИ",

}

# 3. Her sütun için dict map uygulama
for source_col, target_col in map_dict_B3_status.items():
    temp_dict = dict(zip(zmm059_dummy["key"], zmm059_dummy[source_col]))
    df_B3_status[target_col] = df_B3_status["key"].map(temp_dict)

# 4. Geçici key sütunlarını kaldır
zmm059_dummy.drop(columns=["key"], inplace=True)
df_B3_status.drop(columns=["key"], inplace=True)

In [155]:
df_B3_status.columns

Index(['ГОД', 'Заявка', 'Позиция заявки', 'Материал', 'Индикатор удаления_B3',
       'ЗатребованКолич', 'Затребовал', 'Группа деблокир.', 'Дата заявки',
       'Группа закупок', 'Заказанное к-во', 'Единица измерения',
       'RU Наименование', 'TR Наименование', 'Дополнительное',
       'Инвентарный номер', 'Краткий текст', 'еще поставить (количество)',
       'Еще для поставки (стоимость)', 'Для фактурирования (колич.)',
       'Для фактурирования (стоим.)', 'ЗП_Создал', 'Группа материалов',
       'Дата_заявки', 'Материал_ME5A', 'Документ_закупки', 'Зп_Поз',
       'Индикатор удаления', 'имя_поставщика', 'Группа закупок_me2n',
       'Закуп. организация_me2n', 'Наш знак_me2n', 'Ваш код_me2n',
       'Объем заказа_me2n', 'Материал_me2n', 'Складская ЕИ_me2n',
       'ЕИ заказа на постав._me2n', 'Кол-во в ЕИ ввода', 'Дата проводки',
       'Признак 2&кол-во&ЕИ', 'Ссылка&кол-во&ЕИ'],
      dtype='object')

In [156]:
df_B3_status["Закуп. организация_me2n"].sample(n=5)

,Закуп. организация_me2n
385883,2993;nan
383418,nan;2990
349112,nan;2992
89258,nan;2991
288435,2992;nan


In [157]:
df_B3_status.columns

Index(['ГОД', 'Заявка', 'Позиция заявки', 'Материал', 'Индикатор удаления_B3',
       'ЗатребованКолич', 'Затребовал', 'Группа деблокир.', 'Дата заявки',
       'Группа закупок', 'Заказанное к-во', 'Единица измерения',
       'RU Наименование', 'TR Наименование', 'Дополнительное',
       'Инвентарный номер', 'Краткий текст', 'еще поставить (количество)',
       'Еще для поставки (стоимость)', 'Для фактурирования (колич.)',
       'Для фактурирования (стоим.)', 'ЗП_Создал', 'Группа материалов',
       'Дата_заявки', 'Материал_ME5A', 'Документ_закупки', 'Зп_Поз',
       'Индикатор удаления', 'имя_поставщика', 'Группа закупок_me2n',
       'Закуп. организация_me2n', 'Наш знак_me2n', 'Ваш код_me2n',
       'Объем заказа_me2n', 'Материал_me2n', 'Складская ЕИ_me2n',
       'ЕИ заказа на постав._me2n', 'Кол-во в ЕИ ввода', 'Дата проводки',
       'Признак 2&кол-во&ЕИ', 'Ссылка&кол-во&ЕИ'],
      dtype='object')

In [158]:
df_B3_status["ВЗ_Статус"] = pd.NA

In [159]:
mask1 = df_B3_status["Заказанное к-во"].isna() | df_B3_status["Заказанное к-во"].eq(0)
df_B3_status.loc[mask1, "ВЗ_Статус"] = "Заказ поставщику по данной позиции ещё не оформлен"

In [160]:
mask2 = np.isclose(
    df_B3_status["ЗатребованКолич"],
    df_B3_status["Кол-во в ЕИ ввода"],
    rtol=1e-05, atol=1e-08,
    equal_nan=False
)

df_B3_status.loc[mask2, "ВЗ_Статус"] = "Все заказанные материалы поступили на склад и оприходованы."

In [161]:
mask3 = df_B3_status["ЗатребованКолич"].lt(df_B3_status["Кол-во в ЕИ ввода"])
df_B3_status.loc[mask3, "ВЗ_Статус"] = "Оприходование по всем указанным позициям выполнено с толерансом."

In [162]:
mask4 = (
    df_B3_status["ЗатребованКолич"].gt(df_B3_status["Кол-во в ЕИ ввода"])
    & df_B3_status["Кол-во в ЕИ ввода"].ne(0)
)

df_B3_status.loc[mask4, "ВЗ_Статус"] = "Поставка по заказу началась; поставлено частично."

In [163]:
mask5 = (
    df_B3_status["Заказанное к-во"].gt(0)
    & (df_B3_status["Кол-во в ЕИ ввода"].isna() | df_B3_status["Кол-во в ЕИ ввода"].eq(0))
)

df_B3_status.loc[mask5, "ВЗ_Статус"] = "Заказ поставщику создан, но поставка ещё не началась"

In [164]:
display(df_B3_status[df_B3_status["Заявка"]=="2200013039"][["ВЗ_Статус","Заявка","ЗатребованКолич", "Заказанное к-во","Кол-во в ЕИ ввода"]])

,ВЗ_Статус,Заявка,ЗатребованКолич,Заказанное к-во,Кол-во в ЕИ ввода
104415,Поставка по заказу началась; поставлено частично.,2200013039,2971.0,258.824,247.0


In [165]:
df_B3_status["ВЗ_Статус"].fillna("<>").value_counts().reset_index() \
    .rename(columns={"index": "ВЗ_Статус", "ВЗ_Статус": "adet"})

,adet,count
0,Все заказанные материалы поступили на склад и ...,247391
1,Заказ поставщику по данной позиции ещё не офор...,114416
2,"Заказ поставщику создан, но поставка ещё не на...",29189
3,Оприходование по всем указанным позициям выпол...,8540
4,Поставка по заказу началась; поставлено частично.,7577


In [166]:
display(df_B3_status[(df_B3_status["Заявка"]=="2200006369") & (df_B3_status["Позиция заявки"]=="5320")][["Кол-во в ЕИ ввода"]])

,Кол-во в ЕИ ввода
312772,4.0


In [167]:
df_B3_status[["Документ_закупки","Зп_Поз"]].sample(n=5)

,Документ_закупки,Зп_Поз
384818,nan;3100001550,nan;130
152136,nan,nan
8295,nan;3100016117,nan;1230
387552,nan,nan
385620,nan;3100001491,nan;10


In [168]:
zpp001_dummy['Purpose Code'].sample(n=5)

,Purpose Code
1218,Оплата питьевой воды
8811,Металлоконструкции
9489,ABRASİVE MATERİYAL ALIMI
7190,"КМК, 60% после поставки"
726,КАБЕЛЬ /ЭМP/%60 ПОСЛЕ/31-13238


In [169]:
zpp001_dummy.columns

Index(['﻿Валюта оплаты', 'Номер заявки на платеж', 'Тип транзакции',
       'Тип документа', 'Тип платежа', 'Статья ДДС', 'Наименование',
       'Purpose Code', 'Протокол', 'Номер договора', 'Инвойс', 'Кредитор',
       'Название кредитора', 'Оплачиваемая сумма', 'Валюта', 'Автор изменения',
       'Дата изменения', 'Создал', 'Дата создания', 'Unnamed: 19',
       'Статус согласующего', 'Оплаченная Сумма', 'Unnamed: 22', 'Дебитор',
       'Название заказчика', 'Unnamed: 25', 'Документ закупки', 'Unnamed: 27',
       'Unnamed: 28', 'Статус', 'Время', 'Время изменения', 'Дата платежа',
       'fat_ted', 'Номер_счета', 'Дата_документа_счета', 'зп_поз', 'ВЗ',
       'ВЗ_ПОЗ'],
      dtype='object')

In [170]:
zpp001[["Purpose Code","Номер заявки на платеж","Протокол","Номер договора","Инвойс"]].sample(n=5 ,random_state=38)

,Purpose Code,Номер заявки на платеж,Протокол,Номер договора,Инвойс
1251,Электрод/3100014904/ЭМР,100000024984,АК-889,АК-889,PROFORMA FATURA
3269,мат-лы для сварки/ЭМР,100000018277,AK-EM-610,TSM-46-24-282,100037193
3156,Linde GAZ/ SMR/EMR/MMR,100000018603,AK-NP-EM-754,TSM-07-22-859,100018572
8409,Абразивные материалы,100000004487,AK-NP-MM-2376,AK-NP-MM-2376,100045110
9034,Металлоконструкции,100000002900,AK-С-420,AK-С-420,100034083


In [171]:
"""
import re
import pandas as pd

def extract_text_outside_quotes(zp_string):
    if pd.isna(zp_string) or not isinstance(zp_string, str):
        return None

    # Use regex to remove the quoted part (including the quotes)
    # The '.*?' makes the match non-greedy
    text_outside_quotes = re.sub(r'"(.*?)"', '', zp_string)

    # Trim any leading/trailing whitespace from the remaining text
    trimmed_text = text_outside_quotes.strip()

    # Return None if the result is an empty string, otherwise return the trimmed text
    return trimmed_text if trimmed_text else None

def format_zp_numbers(zp_string):
    if pd.isna(zp_string) or not isinstance(zp_string, str) or not zp_string.strip():
        return None

    # Try to extract content within double quotes first, for existing formats
    match_quoted = re.search(r'"([^"]*)"', zp_string)
    if match_quoted:
        # If quotes found, process only the content within them
        content_to_process = match_quoted.group(1)
    else:
        # If no quotes found, clean the whole string and process it
        content_to_process = zp_string.strip()

    # Find all '31' followed by digits, optionally with a hyphen, in the content_to_process
    # This will catch patterns like '31-14677', '3114677', '31-1234'
    # It ensures that only sequences starting with '31' and followed by digits are considered.
    zp_number_patterns = re.findall(r'31-?\d+', content_to_process)

    formatted_numbers = []
    for num_str in zp_number_patterns:
        # Remove any hyphens from the extracted pattern (e.g., '31-14677' -> '3114677')
        clean_num = num_str.replace('-', '')

        if clean_num.startswith('31') and len(clean_num) > 2:
            numeric_part = clean_num[2:] # Get the part after '31'
            if numeric_part.isdigit():
                # Pad with leading zeros to make the part after '31' 8 digits long
                padded_part = numeric_part.zfill(8);
                formatted_numbers.append('31' + padded_part)
            # If numeric_part is not digits, we skip it as it's not a valid ZP number part
        # If it doesn't start with '31' or is too short, we skip it
        # This function focuses specifically on '31' numbers.

    return ';'.join(formatted_numbers) if formatted_numbers else None
    """

<>:35: SyntaxWarning: invalid escape sequence '\d'
<>:35: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_44620/3089327494.py:35: SyntaxWarning: invalid escape sequence '\d'
  zp_number_patterns = re.findall(r'31-?\d+', content_to_process)


'\nimport re\nimport pandas as pd\n\ndef extract_text_outside_quotes(zp_string):\n    if pd.isna(zp_string) or not isinstance(zp_string, str):\n        return None\n\n    # Use regex to remove the quoted part (including the quotes)\n    # The \'.*?\' makes the match non-greedy\n    text_outside_quotes = re.sub(r\'"(.*?)"\', \'\', zp_string)\n\n    # Trim any leading/trailing whitespace from the remaining text\n    trimmed_text = text_outside_quotes.strip()\n\n    # Return None if the result is an empty string, otherwise return the trimmed text\n    return trimmed_text if trimmed_text else None\n\ndef format_zp_numbers(zp_string):\n    if pd.isna(zp_string) or not isinstance(zp_string, str) or not zp_string.strip():\n        return None\n\n    # Try to extract content within double quotes first, for existing formats\n    match_quoted = re.search(r\'"([^"]*)"\', zp_string)\n    if match_quoted:\n        # If quotes found, process only the content within them\n        content_to_process =

In [172]:
"""
# Apply the function to create the new column 'text_zo'
zpp001_dummy['text_zo'] = zpp001_dummy['Purpose Code'].astype(str).apply(extract_text_outside_quotes)

# Apply the updated function to the 'Номер заявки на платеж' column to create/update 'zp no clear'
zpp001_dummy['zp no clear'] = zpp001_dummy['Purpose Code'].astype(str).apply(format_zp_numbers)
"""

"\n# Apply the function to create the new column 'text_zo'\nzpp001_dummy['text_zo'] = zpp001_dummy['Purpose Code'].astype(str).apply(extract_text_outside_quotes)\n\n# Apply the updated function to the 'Номер заявки на платеж' column to create/update 'zp no clear'\nzpp001_dummy['zp no clear'] = zpp001_dummy['Purpose Code'].astype(str).apply(format_zp_numbers)\n"

In [173]:
#zpp001_dummy = zpp001_dummy.drop(columns=['zp no clear', 'text_zo'])

In [174]:
#zpp001_dummy['Номер заявки на платеж'] = zpp001_dummy['Номер заявки на платеж'].apply(temizle_ve_cevir)

In [175]:
#zpp001_dummy.loc[zpp001_dummy['Номер заявки на платеж'] == "100000024260", 'Purpose Code'] = "31-12/31235v31-522/313132 asdvv jfjfj фывв с"

In [176]:
#zpp001_dummy[zpp001_dummy['Номер заявки на платеж']=="100000024260"]

In [177]:
zpp001_dummy.columns

Index(['﻿Валюта оплаты', 'Номер заявки на платеж', 'Тип транзакции',
       'Тип документа', 'Тип платежа', 'Статья ДДС', 'Наименование',
       'Purpose Code', 'Протокол', 'Номер договора', 'Инвойс', 'Кредитор',
       'Название кредитора', 'Оплачиваемая сумма', 'Валюта', 'Автор изменения',
       'Дата изменения', 'Создал', 'Дата создания', 'Unnamed: 19',
       'Статус согласующего', 'Оплаченная Сумма', 'Unnamed: 22', 'Дебитор',
       'Название заказчика', 'Unnamed: 25', 'Документ закупки', 'Unnamed: 27',
       'Unnamed: 28', 'Статус', 'Время', 'Время изменения', 'Дата платежа',
       'fat_ted', 'Номер_счета', 'Дата_документа_счета', 'зп_поз', 'ВЗ',
       'ВЗ_ПОЗ'],
      dtype='object')

In [178]:
"""
columns_to_drop = [
    'Оплачиваемая сумма_zpp001', 'Оплаченная Сумма_zpp001',
    '\ufeffВалюта оплаты_zpp001', 'Тип транзакции_zpp001',
    'Тип документа_zpp001', 'Тип платежа_zpp001', 'Статья ДДС_zpp001',
    'Наименование_zpp001', 'Purpose Code_zpp001', 'Протокол_zpp001',
    'Номер договора_zpp001', 'Инвойс_zpp001', 'Кредитор_zpp001',
    'Название кредитора_zpp001', 'Валюта_zpp001', 'Автор изменения_zpp001',
    'Создал_zpp001', 'Статус согласующего_zpp001', 'Дебитор_zpp001',
    'Название заказчика_zpp001', 'Статус_zpp001', 'text_zo_zpp001',
    'zp no clear_zpp001', 'Дата изменения_zpp001', 'Дата создания_zpp001',
    'Дата платежа_zpp001', 'Время_zpp001', 'Время изменения_zpp001',
    'Документ_закупки_cleaned', 'Номер заявки на платеж_zpp001'
]

df_B3_status.drop(columns=columns_to_drop, inplace=True, errors='ignore')

print("Specified columns dropped from df_B3_status.")
print(f"df_B3_status now has {df_B3_status.shape[1]} columns and {df_B3_status.shape[0]} rows.")
"""

'\ncolumns_to_drop = [\n    \'Оплачиваемая сумма_zpp001\', \'Оплаченная Сумма_zpp001\',\n    \'\ufeffВалюта оплаты_zpp001\', \'Тип транзакции_zpp001\',\n    \'Тип документа_zpp001\', \'Тип платежа_zpp001\', \'Статья ДДС_zpp001\',\n    \'Наименование_zpp001\', \'Purpose Code_zpp001\', \'Протокол_zpp001\',\n    \'Номер договора_zpp001\', \'Инвойс_zpp001\', \'Кредитор_zpp001\',\n    \'Название кредитора_zpp001\', \'Валюта_zpp001\', \'Автор изменения_zpp001\',\n    \'Создал_zpp001\', \'Статус согласующего_zpp001\', \'Дебитор_zpp001\',\n    \'Название заказчика_zpp001\', \'Статус_zpp001\', \'text_zo_zpp001\',\n    \'zp no clear_zpp001\', \'Дата изменения_zpp001\', \'Дата создания_zpp001\',\n    \'Дата платежа_zpp001\', \'Время_zpp001\', \'Время изменения_zpp001\',\n    \'Документ_закупки_cleaned\', \'Номер заявки на платеж_zpp001\'\n]\n\ndf_B3_status.drop(columns=columns_to_drop, inplace=True, errors=\'ignore\')\n\nprint("Specified columns dropped from df_B3_status.")\nprint(f"df_B3_status 

In [179]:
"""
zpp001_dummy.drop(columns='Документ закупки_cleaned', inplace=True, errors='ignore')
zpp001_dummy.drop(columns='zp no clear', inplace=True, errors='ignore')
"""

"\nzpp001_dummy.drop(columns='Документ закупки_cleaned', inplace=True, errors='ignore')\nzpp001_dummy.drop(columns='zp no clear', inplace=True, errors='ignore')\n"

In [180]:
#df_B3_status["zp no clear_zpp001"]

In [181]:
zpp001_dummy.columns

Index(['﻿Валюта оплаты', 'Номер заявки на платеж', 'Тип транзакции',
       'Тип документа', 'Тип платежа', 'Статья ДДС', 'Наименование',
       'Purpose Code', 'Протокол', 'Номер договора', 'Инвойс', 'Кредитор',
       'Название кредитора', 'Оплачиваемая сумма', 'Валюта', 'Автор изменения',
       'Дата изменения', 'Создал', 'Дата создания', 'Unnamed: 19',
       'Статус согласующего', 'Оплаченная Сумма', 'Unnamed: 22', 'Дебитор',
       'Название заказчика', 'Unnamed: 25', 'Документ закупки', 'Unnamed: 27',
       'Unnamed: 28', 'Статус', 'Время', 'Время изменения', 'Дата платежа',
       'fat_ted', 'Номер_счета', 'Дата_документа_счета', 'зп_поз', 'ВЗ',
       'ВЗ_ПОЗ'],
      dtype='object')

In [182]:
import re
import pandas as pd
import numpy as np

# --- Step 1: Define helper functions for ZP number extraction (ensure they are available) ---
def extract_text_outside_quotes(zp_string):
    if pd.isna(zp_string) or not isinstance(zp_string, str):
        return None
    text_outside_quotes = re.sub(r'"(.*?)"', '', zp_string)
    trimmed_text = text_outside_quotes.strip()
    return trimmed_text if trimmed_text else None

def format_zp_numbers(zp_string):
    if pd.isna(zp_string) or not isinstance(zp_string, str) or not zp_string.strip():
        return None
    match_quoted = re.search(r'"([^"]*)"', zp_string)
    content_to_process = match_quoted.group(1) if match_quoted else zp_string.strip()
    zp_number_patterns = re.findall(r'31-?\d+', content_to_process)
    formatted_numbers = []
    for num_str in zp_number_patterns:
        clean_num = num_str.replace('-', '')
        if clean_num.startswith('31') and len(clean_num) > 2:
            numeric_part = clean_num[2:]
            if numeric_part.isdigit():
                padded_part = numeric_part.zfill(8)
                formatted_numbers.append('31' + padded_part)
    return ';'.join(formatted_numbers) if formatted_numbers else None

# --- Step 2: Apply functions to zpp001_dummy to create 'text_zo' and 'zp no clear' ---
zpp001_dummy['text_zo'] = zpp001_dummy['Purpose Code'].astype(str).apply(extract_text_outside_quotes)
zpp001_dummy['zp no clear'] = zpp001_dummy['Purpose Code'].astype(str).apply(format_zp_numbers)

# --- Step 3: Define desired columns for aggregation from zpp001_dummy ---
numeric_cols_zpp001_to_merge = [
    'Оплачиваемая сумма',
    'Оплаченная Сумма'
]
string_cols_zpp001_to_merge = [
    '﻿Валюта оплаты', 'Номер заявки на платеж', 'Наименование', 'Purpose Code',
    'Протокол', 'Номер договора', 'Инвойс', 'Название кредитора', 'Валюта',
    'Автор изменения', 'Создал', 'Дата изменения', 'Дата создания',
    'Статус согласующего', 'Дебитор', 'Название заказчика', 'Статус',
    'text_zo', 'Дата платежа'
]

# Combine all columns to be extracted from zpp001_dummy into zpp001_temp
all_cols_for_temp = ['zp no clear'] + numeric_cols_zpp001_to_merge + string_cols_zpp001_to_merge
zpp001_temp = zpp001_dummy[all_cols_for_temp].copy()

# --- Step 4: Clean numeric columns in zpp001_temp and handle 'zp no clear' for explode ---
for col_name in numeric_cols_zpp001_to_merge:
    zpp001_temp[col_name] = (
        zpp001_temp[col_name]
        .astype(str)
        .str.replace(" ", "", regex=False)
        .str.replace(".", "", regex=False)
        .str.replace(",", ".", regex=False)
        .astype(float)
    )

# Ensure 'zp no clear' is a list of strings for explode, handling NA/empty values
zpp001_temp['zp no clear'] = zpp001_temp['zp no clear'].astype(str).apply(
    lambda x: x.split(';') if pd.notna(x) and x.strip() != '' and x.strip().lower() != 'nan' else []
)

# --- Step 5: Explode and filter zpp001_temp ---
zpp001_exploded = zpp001_temp.explode('zp no clear')
zpp001_exploded = zpp001_exploded[zpp001_exploded['zp no clear'].astype(str).str.strip() != ''].copy()

# --- Step 6: Define aggregation dictionary and create zpp001_lookup_df ---
agg_dict_for_lookup = {}
for col in numeric_cols_zpp001_to_merge:
    agg_dict_for_lookup[col] = 'sum'
for col in string_cols_zpp001_to_merge:
    agg_dict_for_lookup[col] = lambda x: '; '.join(x.dropna().astype(str).unique())

zpp001_lookup_df = zpp001_exploded.groupby('zp no clear', as_index=False).agg(agg_dict_for_lookup)

# --- Step 7: Redefine clean_document_numbers (ensure it's available) ---
def clean_document_numbers(doc_num):
    if pd.isna(doc_num):
        return pd.NA
    doc_str = str(doc_num).strip()
    if not doc_str or doc_str.lower() == 'nan':
        return pd.NA
    doc_str_cleaned = doc_str.replace('-', '')
    potential_numbers = re.findall(r'31\d+', doc_str_cleaned)
    cleaned_numbers = []
    for num_str in potential_numbers:
        if num_str.startswith('31') and len(num_str) > 2:
            numeric_part = num_str[2:]
            if numeric_part.isdigit():
                padded_part = numeric_part.zfill(8)
                if len(padded_part) > 8:
                    padded_part = padded_part[-8:]
                cleaned_numbers.append('31' + padded_part)
    return ';'.join(cleaned_numbers) if cleaned_numbers else pd.NA

# --- Step 8: Apply clean_document_numbers to df_B3_status ---
df_B3_status['Документ_закупки_cleaned'] = df_B3_status['Документ_закупки'].apply(clean_document_numbers)

# --- Step 9: Perform the merge loop and clean up ---

# Initialize new columns in df_B3_status with NaN/NA
for col in string_cols_zpp001_to_merge:
    df_B3_status[col + '_zpp001'] = pd.NA
for col in numeric_cols_zpp001_to_merge:
    df_B3_status[col + '_zpp001'] = np.nan

# Iterate through df_B3_status to merge data
for index, row in df_B3_status.iterrows():
    cleaned_doc_nums_str = row['Документ_закупки_cleaned']
    if pd.notna(cleaned_doc_nums_str):
        doc_nums = str(cleaned_doc_nums_str).split(';')

        # Filter lookup DataFrame for matching document numbers
        matching_rows = zpp001_lookup_df[zpp001_lookup_df['zp no clear'].isin(doc_nums)]

        if not matching_rows.empty:
            # Aggregate string columns by unique concatenation
            for col in string_cols_zpp001_to_merge:
                all_values = set()
                for item in matching_rows[col].dropna():
                    for sub_item in str(item).split(';'):
                        if sub_item.strip():
                            all_values.add(sub_item.strip())
                if all_values:
                    df_B3_status.loc[index, col + '_zpp001'] = ';'.join(sorted(list(all_values)))
                else:
                    df_B3_status.loc[index, col + '_zpp001'] = pd.NA

            # Aggregate numeric columns by summing
            for col in numeric_cols_zpp001_to_merge:
                sum_val = matching_rows[col].sum()
                if pd.notna(sum_val): # Assign if sum is a number (including 0.0)
                    df_B3_status.loc[index, col + '_zpp001'] = sum_val
                else: # Only assign NaN if the sum itself was NaN (e.g., all values were NaN)
                    df_B3_status.loc[index, col + '_zpp001'] = np.nan

# Drop the temporary 'Документ_закупки_cleaned' column
df_B3_status.drop(columns=['Документ_закупки_cleaned'], inplace=True)

print("zpp001 data has been successfully merged into df_B3_status.")
print(f"df_B3_status now has {df_B3_status.shape[1]} columns and {df_B3_status.shape[0]} rows.")

zpp001 data has been successfully merged into df_B3_status.
df_B3_status now has 63 columns and 407113 rows.


In [183]:
df_B3_status.columns

Index(['ГОД', 'Заявка', 'Позиция заявки', 'Материал', 'Индикатор удаления_B3',
       'ЗатребованКолич', 'Затребовал', 'Группа деблокир.', 'Дата заявки',
       'Группа закупок', 'Заказанное к-во', 'Единица измерения',
       'RU Наименование', 'TR Наименование', 'Дополнительное',
       'Инвентарный номер', 'Краткий текст', 'еще поставить (количество)',
       'Еще для поставки (стоимость)', 'Для фактурирования (колич.)',
       'Для фактурирования (стоим.)', 'ЗП_Создал', 'Группа материалов',
       'Дата_заявки', 'Материал_ME5A', 'Документ_закупки', 'Зп_Поз',
       'Индикатор удаления', 'имя_поставщика', 'Группа закупок_me2n',
       'Закуп. организация_me2n', 'Наш знак_me2n', 'Ваш код_me2n',
       'Объем заказа_me2n', 'Материал_me2n', 'Складская ЕИ_me2n',
       'ЕИ заказа на постав._me2n', 'Кол-во в ЕИ ввода', 'Дата проводки',
       'Признак 2&кол-во&ЕИ', 'Ссылка&кол-во&ЕИ', 'ВЗ_Статус',
       '﻿Валюта оплаты_zpp001', 'Номер заявки на платеж_zpp001',
       'Наименование_zpp001

In [184]:
df_B3_status[['Дата платежа_zpp001','Номер заявки на платеж_zpp001']].sample(n=5)

,Дата платежа_zpp001,Номер заявки на платеж_zpp001
182104,<NA>,<NA>
360180,<NA>,<NA>
299978,<NA>,<NA>
388943,<NA>,<NA>
389466,<NA>,<NA>


In [185]:
df_B3_status["Оплачиваемая сумма_zpp001"].sample(n=5)

,Оплачиваемая сумма_zpp001
88553,NaN
90023,NaN
204327,NaN
15197,NaN
219651,NaN


In [186]:
df_B3_status.columns

Index(['ГОД', 'Заявка', 'Позиция заявки', 'Материал', 'Индикатор удаления_B3',
       'ЗатребованКолич', 'Затребовал', 'Группа деблокир.', 'Дата заявки',
       'Группа закупок', 'Заказанное к-во', 'Единица измерения',
       'RU Наименование', 'TR Наименование', 'Дополнительное',
       'Инвентарный номер', 'Краткий текст', 'еще поставить (количество)',
       'Еще для поставки (стоимость)', 'Для фактурирования (колич.)',
       'Для фактурирования (стоим.)', 'ЗП_Создал', 'Группа материалов',
       'Дата_заявки', 'Материал_ME5A', 'Документ_закупки', 'Зп_Поз',
       'Индикатор удаления', 'имя_поставщика', 'Группа закупок_me2n',
       'Закуп. организация_me2n', 'Наш знак_me2n', 'Ваш код_me2n',
       'Объем заказа_me2n', 'Материал_me2n', 'Складская ЕИ_me2n',
       'ЕИ заказа на постав._me2n', 'Кол-во в ЕИ ввода', 'Дата проводки',
       'Признак 2&кол-во&ЕИ', 'Ссылка&кол-во&ЕИ', 'ВЗ_Статус',
       '﻿Валюта оплаты_zpp001', 'Номер заявки на платеж_zpp001',
       'Наименование_zpp001

In [187]:
# 1. Yeni sütun sırasını belirle
B3status_yenisira = [
    'ГОД', 'Заявка','Позиция заявки','ВЗ_Статус','Дата заявки','Материал','Индикатор удаления_B3','ЗатребованКолич','Единица измерения',
    'Затребовал','Группа закупок','RU Наименование','TR Наименование','Дополнительное','Краткий текст','Инвентарный номер',
    'Дата_заявки','Группа материалов','Группа деблокир.',
    'Заказанное к-во',
    'Документ_закупки','Зп_Поз','ЗП_Создал','Индикатор удаления','имя_поставщика','Наш знак_me2n','Ваш код_me2n',
    'Объем заказа_me2n','ЕИ заказа на постав._me2n','Складская ЕИ_me2n','еще поставить (количество)','Еще для поставки (стоимость)',
    'Для фактурирования (колич.)','Для фактурирования (стоим.)','Группа закупок_me2n','Закуп. организация_me2n',
    'Кол-во в ЕИ ввода','Дата проводки','Признак 2&кол-во&ЕИ','Ссылка&кол-во&ЕИ',
    'Номер заявки на платеж_zpp001','Purpose Code_zpp001','text_zo_zpp001','Статус согласующего_zpp001','Оплачиваемая сумма_zpp001','Валюта_zpp001','Дата платежа_zpp001',
    'Оплаченная Сумма_zpp001','\ufeffВалюта оплаты_zpp001','Протокол_zpp001','Номер договора_zpp001','Создал_zpp001',
    'Дата создания_zpp001','Дата изменения_zpp001','Автор изменения_zpp001','Название кредитора_zpp001',
    'Наименование_zpp001','Инвойс_zpp001','Дебитор_zpp001','Название заказчика_zpp001'

]

# 2. DataFrame'i yeni sütun sırasına göre düzenle
df_B3_status = df_B3_status[B3status_yenisira]


In [188]:
df_calisilan["Документ_закупки"] = df_calisilan["Документ_закупки"].apply(temizle_ve_cevir)
df_calisilan["Зп_Поз"] = df_calisilan["Зп_Поз"].apply(temizle_ve_cevir)
df_calisilan["Заявка"] = df_calisilan["Заявка"].apply(temizle_ve_cevir)
df_calisilan["Позиция заявки"] = df_calisilan["Позиция заявки"].apply(temizle_ve_cevir)

In [189]:
# 1. Anahtar sütunları tuple haline getirelim
df_calisilan["key"] = df_calisilan[["Документ_закупки","Зп_Поз"]].apply(tuple, axis=1)
df_zmm070_2022_2025["key"] = df_zmm070_2022_2025[["Документ закупки","Позиция"]].apply(tuple, axis=1)

# 2. Aktarılacak sütunlar ve hedef adları
map_dict_B3_status = {
    "Заявка":"Заявка",
    "Позиция заявки":"Позиция заявки"
}

# 3. Her sütun için dict map uygulama
for source_col, target_col in map_dict_B3_status.items():
    temp_dict = dict(zip(df_calisilan["key"], df_calisilan[source_col]))
    df_zmm070_2022_2025[target_col] = df_zmm070_2022_2025["key"].map(temp_dict)

# 4. Geçici key sütunlarını kaldır
df_calisilan.drop(columns=["key"], inplace=True)
df_zmm070_2022_2025.drop(columns=["key"], inplace=True)

In [190]:
# Gruplama için kullanılacak sütunlar
group_cols = ['Документ закупки', 'Позиция']
# Aggregation (toplama/birleştirme) fonksiyonları
agg_funcs = {
    'Ссылка': lambda x: ';'.join(x.dropna().astype(str)),
    '﻿Fatıra Tarihi': lambda x: ';'.join(x.dropna().astype(str)),
}

# Gruplama ve aggregation işlemini gerçekleştiriyoruz
result_df = df_zmm070_2022_2025.groupby(group_cols, as_index=False).agg(agg_funcs)
df_zmm070_2022_2025_dummyzp=result_df.copy()
del result_df

In [191]:
df_zmm070_2022_2025_dummyzp.sample(n=5)

,Документ закупки,Позиция,Ссылка,﻿Fatıra Tarihi
67788,3100009246,3250,L012026000000060,06.03.2026
154196,3100015117,170,HIL2026906404575;HIL2026906404575,31.03.2026;31.03.2026
113022,3100011638,140,SN12025000000006,14.01.2025
59738,3100009002,2970,TSI2026000000084;L012024000000321,7.02.2026;1.08.2024
126947,3100012743,180,L012024000000092,31.03.2024


In [192]:
df_me2n_2022_2025.columns

Index(['ГОД', 'Документ закупки', 'Позиция', 'Создал', 'Группа закупок',
       'Закуп. организация', 'Индикатор удаления', 'Дата документа',
       'Наш знак', 'Ваш код', 'Имя поставщика', 'Заявка', 'Позиция заявки',
       'Материал', 'Инвентарный номер', 'Наименование RU', 'Наименование TR',
       'Объем заказа', 'Складская ЕИ', 'еще поставить (количество)',
       'Еще для поставки (стоимость)', 'Уже поставлено (количество)',
       'Для фактурирования (стоим.)', 'Для фактурирования (колич.)',
       'Количество в СЕИ', 'Единица цены', 'СтоимЗаказа нетто', 'Валюта',
       'Цена нетто', 'Вид докум. закупки', 'Код налога', 'Краткий текст',
       'Дата поставки', 'ЕИ заказа на постав.', 'ТипДокумЗакупки',
       'Тип неполноты данных', 'Группа материалов', 'Материал_ME2N',
       'Материал_ME5A'],
      dtype='object')

In [193]:
df_zmm070_2022_2025_dummyzp.sample(n=10)

,Документ закупки,Позиция,Ссылка,﻿Fatıra Tarihi
19675,3100006158,70,MET2024000000051,7.02.2024
129782,3100012843,10,L012026000000032;TSI2026000000084;L01202500000...,13.02.2026;7.02.2026;30.09.2025
99718,3100011190,40,ODY2025000000005;ODY2025000000004,27.01.2025;27.01.2025
90744,3100010499,90,L012024000000496,9.10.2024
83500,3100009865,860,L012024000000417,6.09.2024
58878,3100008897,490,7482/1958,22.07.2024
154244,3100015130,10,AYM2026000000388;AYM2026000000428;AYM202600000...,31.03.2026;07.04.2026;07.04.2026;31.03.2026;07...
148107,3100014424,30,GM12026000000014,03.04.2026
106524,3100011254,44300,J02202400000298;J02202400000298,01.09.2024;01.09.2024
70068,3100009298,1360,TSI2026000000084;L012024000000321,7.02.2026;1.08.2024


In [194]:
df_me2n_2022_2025["Документ закупки"] = df_me2n_2022_2025["Документ закупки"].apply(temizle_ve_cevir)

In [195]:
df_me2n_2022_2025["Позиция"] = df_me2n_2022_2025["Позиция"].apply(temizle_ve_cevir)

In [196]:
# 1. Anahtar sütunları tuple haline getirelim

df_zmm070_2022_2025_dummyzp["key"] = df_zmm070_2022_2025_dummyzp[["Документ закупки","Позиция"]].apply(tuple, axis=1)
df_me2n_2022_2025["key"] = df_me2n_2022_2025[["Документ закупки","Позиция"]].apply(tuple, axis=1)

# 2. Aktarılacak sütunlar ve hedef adları
map_dict_zp_fatura_status = {
    "Ссылка":"счет фактура",
    "﻿Fatıra Tarihi":"Дата выставления счета",
}

# 3. Her sütun için dict map uygulama
for source_col, target_col in map_dict_zp_fatura_status.items():
    temp_dict = dict(zip(df_zmm070_2022_2025_dummyzp["key"], df_zmm070_2022_2025_dummyzp[source_col]))
    df_me2n_2022_2025[target_col] = df_me2n_2022_2025["key"].map(temp_dict)

# 4. Geçici key sütunlarını kaldır
df_zmm070_2022_2025_dummyzp.drop(columns=["key"], inplace=True)
df_me2n_2022_2025.drop(columns=["key"], inplace=True)

In [197]:
df_me2n_2022_2025.loc[(df_me2n_2022_2025["Документ закупки"] == "3100005585") & (df_me2n_2022_2025["Позиция"] == "370"), ["Документ закупки","Позиция", "счет фактура"]]

,Документ закупки,Позиция,счет фактура
300508,3100005585,370,7482/1851


In [198]:
df_me2n_2022_2025.columns

Index(['ГОД', 'Документ закупки', 'Позиция', 'Создал', 'Группа закупок',
       'Закуп. организация', 'Индикатор удаления', 'Дата документа',
       'Наш знак', 'Ваш код', 'Имя поставщика', 'Заявка', 'Позиция заявки',
       'Материал', 'Инвентарный номер', 'Наименование RU', 'Наименование TR',
       'Объем заказа', 'Складская ЕИ', 'еще поставить (количество)',
       'Еще для поставки (стоимость)', 'Уже поставлено (количество)',
       'Для фактурирования (стоим.)', 'Для фактурирования (колич.)',
       'Количество в СЕИ', 'Единица цены', 'СтоимЗаказа нетто', 'Валюта',
       'Цена нетто', 'Вид докум. закупки', 'Код налога', 'Краткий текст',
       'Дата поставки', 'ЕИ заказа на постав.', 'ТипДокумЗакупки',
       'Тип неполноты данных', 'Группа материалов', 'Материал_ME2N',
       'Материал_ME5A', 'счет фактура', 'Дата выставления счета'],
      dtype='object')

In [199]:
df_zmm070_2022_2025.columns

Index(['﻿Fatıra Tarihi', 'Kayıt Tarihi', 'Ссылка', 'Сумма счета брутто',
       'Валюта', 'Сумма налога', 'Текст', 'Выставитель счета', 'Имя 1', 'БЕ',
       'Позиция счета', 'Сумма', 'Количество', 'ЕИ заказа на постав.',
       'Документ закупки', 'Позиция', 'Код налога', 'Тип контировки',
       'Бизнес-сфера', 'Краткий текст', 'Счет Главной книги',
       'Основное средство', 'Unnamed: 22', 'Цена заказа нетто', 'Заявка',
       'Позиция заявки'],
      dtype='object')

In [200]:
# Gruplama için kullanılacak sütunlar
group_cols = ['Заявка', 'Позиция заявки']
# Aggregation (toplama/birleştirme) fonksiyonları
agg_funcs = {
    'Ссылка': lambda x: ';'.join(x.dropna().astype(str)),
    '﻿Fatıra Tarihi': lambda x: ';'.join(x.dropna().astype(str)),
}

# Gruplama ve aggregation işlemini gerçekleştiriyoruz
result_df = df_zmm070_2022_2025.groupby(group_cols, as_index=False).agg(agg_funcs)
df_zmm070_2022_2025_dummyvz=result_df.copy()
del result_df

In [ ]:
zpp001.to_excel("zpp001.xlsx", index=False)
files.download("zpp001.xlsx")

In [ ]:
df_B3_status.to_excel("B3_status-2026-2022.xlsx", index=False)
files.download("B3_status-2026-2022.xlsx")

In [ ]:
df_me5a_cleaned.to_excel("df_me5a_cleaned.xlsx", index=False)
files.download("df_me5a_cleaned.xlsx")

In [ ]:
df_zmm067_2022_2025.to_excel("zmm067_2026_2022.xlsx", index=False)
files.download("zmm067_2026_2022.xlsx")

In [ ]:
df_me2n_2022_2025_analog.to_excel("me2n_2026_2022_analog_pos.xlsx", index=False)
files.download("me2n_2026_2022_analog_pos.xlsx")

In [ ]:
df_me5a_2022_2025.to_excel("me5a_2026_2022.xlsx", index=False)
files.download("me5a_2026_2022.xlsx")

In [ ]:
df_me2n_2022_2025.to_excel("me2n_2026_2022.xlsx", index=False)
files.download("me2n_2026_2022.xlsx")

In [ ]:
birlesik_df_zmb51.to_excel("zmb51_2026_2022.xlsx", index=False)
files.download("zmb51_2026_2022.xlsx")

In [ ]:
df_calisilan.to_excel("zmm059_2026_2022.xlsx", index=False, engine="xlsxwriter")
files.download("zmm059_2026_2022.xlsx")